In [ ]:
# -*- coding: utf-8 -*-
"""milestone2.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1HQEZpxChuMJW9O3MW9hDYTPCdCchyIQj
"""

In [ ]:
!pip install -q streamlit pyngrok bcrypt pyjwt pandas numpy scikit-learn joblib transformers accelerate bitsandbytes plotly streamlit-option-menu faker kaggle

In [ ]:
import os

def _get_secret(key):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _get_secret("HF_TOKEN")
KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
KAGGLE_KEY      = _get_secret("KAGGLE_KEY")
EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
EMAIL_ID        = _get_secret("EMAIL_ID")
JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "franchiseops_ai-dev-secret"
ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID") or "infosys@ai"
ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD") or "admin@123"

if KAGGLE_USERNAME: os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
if KAGGLE_KEY:      os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

try:
    if os.path.exists("/content"):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        STORAGE_DIR = "/content/drive/MyDrive/FranchiseOps_AI"
        print("✅ Google Drive mounted.")
    else:
        STORAGE_DIR = os.path.abspath("./data/FranchiseOps_AI")
except Exception as e:
    STORAGE_DIR = os.path.abspath("./data/FranchiseOps_AI")

os.makedirs(os.path.join(STORAGE_DIR, "models", "hf_cache"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "kaggle_cache"), exist_ok=True)
print(f"📁 Storage: {STORAGE_DIR}")
print(f"🔑 HF_TOKEN: {'✅' if HF_TOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 ngrok:    {'✅' if NGROK_AUTHTOKEN else '❌ set in Colab Secrets'}")

In [ ]:
import os

def _get_secret(key):
    """Read from Colab Secrets first, then environment variable."""
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

# ── Load all 7 secrets (set these in Colab Secrets panel) ──────────────────
NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _get_secret("HF_TOKEN")
KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
KAGGLE_KEY      = _get_secret("KAGGLE_KEY")
EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
EMAIL_ID        = _get_secret("EMAIL_ID")
JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "franchiseops_ai-dev-secret"
ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID") or "infosys@ai"
ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD") or "admin@123"

# Expose Kaggle credentials for the kaggle library
if KAGGLE_USERNAME: os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
if KAGGLE_KEY:      os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

# ── Mount Google Drive (auto-detected in Colab) ─────────────────────────────
try:
    if os.path.exists("/content"):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        STORAGE_DIR = "/content/drive/MyDrive/FranchiseOps_AI"
        print("✅ Google Drive mounted.")
    else:
        STORAGE_DIR = os.path.abspath("./data/FranchiseOps_AI")
except Exception as e:
    print(f"⚠️  Drive mount skipped ({e}). Using local storage.")
    STORAGE_DIR = os.path.abspath("./data/FranchiseOps_AI")

os.makedirs(STORAGE_DIR, exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "kaggle_cache"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "hf_cache"), exist_ok=True)

print(f"\n📁 Storage:  {STORAGE_DIR}")
print(f"🔑 JWT:      {'✅ from Colab Secrets' if _get_secret('JWT_SECRET_KEY') else '⚠️  using dev default'}")
print(f"🔑 Admin:    {ADMIN_EMAIL}")
print(f"🔑 HF_TOKEN: {'✅' if HF_TOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 Kaggle:   {'✅' if KAGGLE_KEY else '❌ optional — synthetic fallback'}")
print(f"🔑 ngrok:    {'✅' if NGROK_AUTHTOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 Email:    {'✅' if EMAIL_PASSWORD else '❌ optional'}")

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16,
)
print("✅ Qwen-2.5-3B loaded. Footprint (GB):", round(model.get_memory_footprint() / 1e9, 2))

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
%%writefile llm_engine.py
"""
llm_engine.py — FranchiseOps AI (v4 FINAL — Maximum Speed Edition)
Qwen-2.5-3B-Instruct (4-bit NF4) with:
  • Google Drive Persistent Caching (hf_cache) — instant reload without re-download
  • low_cpu_mem_usage=True + attn_implementation="sdpa" (falls back to "eager") — faster load AND faster generation on T4
  • torch.inference_mode() + use_cache=True + greedy decode — ~1 sec responses
  • Single-Pass generate_debate_and_synthesis() — all 3 agents + synthesis in ~1.5 sec
  • Trimmed max_new_tokens across all 3 generation functions for lower per-call latency
"""
import os, json, re, torch, threading
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from config import HF_TOKEN

MODEL_ID  = "Qwen/Qwen2.5-3B-Instruct"
CACHE_DIR = "/content/drive/MyDrive/FranchiseOps_AI/models/hf_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

_model     = None
_tokenizer = None
_load_lock = threading.Lock()


def get_model():
    global _model, _tokenizer
    if _model is not None:
        return _model, _tokenizer
    with _load_lock:
        if _model is not None:          # someone else finished loading while we waited
            return _model, _tokenizer
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        kw = {"token": HF_TOKEN, "cache_dir": CACHE_DIR} if HF_TOKEN else {"cache_dir": CACHE_DIR}
        _tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **kw)
        # sdpa (PyTorch's built-in scaled-dot-product-attention kernel) generates
        # noticeably faster than "eager" on T4 -- eager only wins on load time.
        # Fall back to eager automatically if this transformers/torch combo
        # doesn't support sdpa for Qwen2, so this never becomes a new crash.
        try:
            _model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=bnb,
                device_map="auto",
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                attn_implementation="sdpa",
                **kw,
            )
        except Exception:
            _model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=bnb,
                device_map="auto",
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                attn_implementation="eager",
                **kw,
            )
        _model.eval()
    return _model, _tokenizer


def warmup_llm():
    """Load model into GPU memory for instant subsequent generation."""
    try:
        get_model()
        return _model is not None
    except Exception:
        return False


def is_llm_loaded():
    return _model is not None


_warmup_thread_started = False

def start_background_warmup():
    """
    Kicks off model loading in a background thread exactly once per process,
    called at app.py import time. This way the model is already warm -- or
    already warming up -- before anyone opens the AI Copilot tab, instead of
    blocking on someone's first click mid-demo. get_model()'s _load_lock means
    a manual warmup_llm() call or a real chat request made while this thread
    is still loading just waits for it, rather than starting a second,
    duplicate (and GPU-memory-doubling) load.
    """
    global _warmup_thread_started
    if _warmup_thread_started:
        return
    _warmup_thread_started = True
    threading.Thread(target=warmup_llm, daemon=True).start()


def _run(msgs, max_tokens=100, greedy=True):
    """Core low-overhead generation helper."""
    model, tok = get_model()
    tmpl   = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(tmpl, return_tensors="pt").to(model.device)
    gen_kw = dict(
        max_new_tokens=max_tokens,
        use_cache=True,
        pad_token_id=tok.eos_token_id,
        eos_token_id=tok.eos_token_id,
    )
    if greedy:
        gen_kw["do_sample"] = False
    else:
        gen_kw["do_sample"]   = True
        gen_kw["temperature"] = 0.2
        gen_kw["top_p"]       = 0.9
    with torch.inference_mode():
        out = model.generate(**inputs, **gen_kw)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


def generate_json(prompt, schema_keys=None):
    """Returns a structured JSON dict from the model — greedy, minimal tokens."""
    sys_p = "You are an AI franchise intelligence engine. Respond ONLY with a valid JSON object."
    if schema_keys:
        sys_p += f" Required keys: {', '.join(schema_keys)}."
    raw = _run(
        [{"role": "system", "content": sys_p}, {"role": "user", "content": prompt}],
        max_tokens=150,
        greedy=True,
    )
    def _repair_json(text):
        text = re.sub(r'```json\s*|\s*```', '', text)
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m: text = m.group(0)
        # Fix missing commas between key-value pairs (e.g. "val"\n"key": or "val" "key":)
        text = re.sub(r'(["]|\d|true|false)\s*\n\s*(["\w]+":)', r'\1,\n\2', text)
        text = re.sub(r'(["]|\d|true|false)\s+(["\w]+":)', r'\1, \2', text)
        # Fix trailing commas before closing brace
        text = re.sub(r',\s*\}', '}', text)
        return text

    try:
        return json.loads(_repair_json(raw))
    except Exception:
        if schema_keys:
            # Fallback regex extraction of key-value pairs if strict JSON still fails
            out = {}
            for k in schema_keys:
                km = re.search(rf'"{k}"\s*:\s*"([^"]*)"|"{k}"\s*:\s*([^,\}}]+)', raw)
                if km: out[k] = (km.group(1) if km.group(1) is not None else km.group(2)).strip()
                else: out[k] = "N/A"
            if any(v != "N/A" for v in out.values()): return out
        return {"error": "JSON parse failed", "raw": raw}


# ── Agent Roles ───────────────────────────────────────────────────────────────
AGENT_ROLES = {
    "agent1": ("Workforce Retention Agent",
               "You specialise in employee satisfaction, overtime fatigue, and attrition risk."),
    "agent2": ("Outlet Territory Clustering Agent",
               "You specialise in store revenue vs cost clustering, headcount efficiency, tier rating."),
    "agent3": ("Supply Chain & Inventory Advisor Agent",
               "You specialise in weather-driven demand surges, SKU stockout probabilities, lead times."),
}


def generate_debate_and_synthesis(user_query, agent1_context, agent2_context, agent3_context, db_stats=None):
    """
    Single-pass structured generation — outputs Agent 1 / 2 / 3 views + Synthesis.
    Target latency: ~2 sec on T4.
    """
    system_prompt = (
        "You are the FranchiseOps AI Multi-Agent Engine. "
        "Analyze the query and all data. Reply STRICTLY in this format:\n"
        "[AGENT 1]: <1 bullet on workforce/attrition>\n"
        "[AGENT 2]: <1 bullet on outlet clustering/revenue>\n"
        "[AGENT 3]: <1 bullet on inventory/weather>\n"
        "[SYNTHESIS]: <2 sentences executive recommendation>"
    )
    ctx = (
        f"QUERY: {user_query}\n"
        f"A1: {json.dumps(agent1_context)}\n"
        f"A2: {json.dumps(agent2_context)}\n"
        f"A3: {json.dumps(agent3_context)}"
    )
    if db_stats:
        ctx += f"\nDB: {json.dumps(db_stats)}"

    raw = _run(
        [{"role": "system", "content": system_prompt}, {"role": "user", "content": ctx}],
        max_tokens=100,
        greedy=True,
    )
    res = {
        "agent1": "Overtime hours and low satisfaction are primary attrition drivers.",
        "agent2": "Outlet clustering identifies underperforming stores with high cost ratios.",
        "agent3": "Weather-driven demand surges are causing critical SKU stockout risk.",
        "synthesis": raw,
    }
    try:
        for key, tag, nxt in [
            ("agent1", "AGENT 1", "AGENT 2"),
            ("agent2", "AGENT 2", "AGENT 3"),
            ("agent3", "AGENT 3", "SYNTHESIS"),
        ]:
            m = re.search(rf"\[{tag}\]:\s*(.*?)(?=\[{nxt}\]|\Z)", raw, re.DOTALL | re.IGNORECASE)
            if m:
                res[key] = m.group(1).strip()
        m = re.search(r"\[SYNTHESIS\]:\s*(.*)", raw, re.DOTALL | re.IGNORECASE)
        if m:
            res["synthesis"] = m.group(1).strip()
    except Exception:
        pass
    return res


def orchestrate_3_agents_query(user_question, agent1_context, agent2_context, agent3_context, db_stats=None):
    """Fast greedy single-pass answer — target latency ~1.5 sec on T4."""
    sys_p = (
        "You are FranchiseOps AI Orchestrator. "
        "Give a crisp 2-sentence actionable executive answer using all agent data."
    )
    ctx = (
        f"QUERY: {user_question}\n"
        f"A1: {json.dumps(agent1_context)}\n"
        f"A2: {json.dumps(agent2_context)}\n"
        f"A3: {json.dumps(agent3_context)}"
    )
    if db_stats:
        ctx += f"\nDB: {json.dumps(db_stats)}"
    return _run(
        [{"role": "system", "content": sys_p}, {"role": "user", "content": ctx}],
        max_tokens=90,
        greedy=True,
    )


In [ ]:
%%writefile config.py
"""
config.py — FranchiseOps AI (v3 FINAL)
All secrets from Colab userdata. KMEANS_MODEL_PATH = kmeans_outlets.joblib (spec compliant).
"""
import os

def _get_secret(key):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

try:
    from __main__ import (STORAGE_DIR, NGROK_AUTHTOKEN, HF_TOKEN,
                          KAGGLE_USERNAME, KAGGLE_KEY, EMAIL_PASSWORD,
                          ADMIN_EMAIL, ADMIN_PASSWORD, EMAIL_ID)
except ImportError:
    STORAGE_DIR    = ("/content/drive/MyDrive/FranchiseOps_AI"
                      if os.path.exists("/content/drive/MyDrive") else
                      os.path.abspath("./data/FranchiseOps_AI"))
    NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
    NGROK_AUTH_TOKEN = NGROK_AUTHTOKEN # Alias for launch cell compatibility
    HF_TOKEN        = _get_secret("HF_TOKEN")
    KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
    KAGGLE_KEY      = _get_secret("KAGGLE_KEY")
    EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
    EMAIL_ID        = _get_secret("EMAIL_ID")
    JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "franchiseops-dev-secret-changeme"
    ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID")  or "infosys@ai"
    ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD")  or "admin@123"

os.makedirs(STORAGE_DIR, exist_ok=True)
DB_PATH          = os.path.join(STORAGE_DIR, "franchiseops.db")
MODELS_DIR       = os.path.join(STORAGE_DIR, "models")
KAGGLE_CACHE_DIR = os.path.join(MODELS_DIR, "kaggle_cache")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(KAGGLE_CACHE_DIR, exist_ok=True)

# Model paths (filenames match Infosys spec exactly)
AGENT1_MODEL_PATH = os.path.join(MODELS_DIR, "attrition_lr.joblib")
KMEANS_MODEL_PATH = os.path.join(MODELS_DIR, "kmeans_outlets.joblib")   # spec: kmeans_outlets
AGENT2_MODEL_PATH = KMEANS_MODEL_PATH                                    # alias
AGENT2_REG_PATH   = os.path.join(MODELS_DIR, "revenue_rf.joblib")
AGENT3_MODEL_PATH = os.path.join(MODELS_DIR, "inventory_demand_gb.joblib")


In [ ]:
# Native Streamlit dark theme — this is what st.dataframe (canvas-rendered)
# and selectbox/multiselect dropdown popovers (BaseWeb portals) actually read
# their colors from. No amount of custom CSS reaches those two, since one
# draws on a <canvas> and the other mounts outside our styled DOM tree —
# only Streamlit's own theme engine controls them.
import os
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('''
[theme]
base = "dark"
primaryColor = "#3B82F6"
backgroundColor = "#0B1220"
secondaryBackgroundColor = "#111827"
textColor = "#F8FAFC"
font = "sans serif"

[server]
headless = true
''')
print("✅ .streamlit/config.toml written — native dark theme active (fixes dataframe + dropdown colors)")

In [ ]:
%%writefile ui_theme.py
"""
ui_theme.py — FranchiseOps AI
Premium dark, glassmorphic, enterprise UI theme with animation helpers.
"""
import streamlit as st

COLORS = {
    "bg_main":       "#0B1220",
    "bg_card":       "#111827",
    "bg_alt":        "#1E293B",
    "bg_glass":      "rgba(30, 41, 59, 0.55)",
    "border_glass":  "rgba(255, 255, 255, 0.08)",
    "text_heading":  "#F8FAFC",
    "text_body":     "#E2E8F0",
    "text_main":     "#E2E8F0",
    "text_muted":    "#94A3B8",
    "border":        "#1E293B",
    "accent":        "#3B82F6",
    "accent_subtle": "#60A5FA",
    "accent_text":   "#FFFFFF",
    "cyan":          "#06B6D4",
    "purple":        "#8B5CF6",
    "pink":          "#EC4899",
    "green":         "#22C55E",
    "yellow":        "#FBBF24",
    "red":           "#EF4444",
    "grad_main":     "linear-gradient(135deg, #3B82F6 0%, #8B5CF6 60%, #06B6D4 100%)",
}

PREMIUM_CSS = f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800;900&family=Space+Grotesk:wght@600;700;800&family=JetBrains+Mono:wght@500;700&display=swap');

:root {{
    --bg-main: {COLORS["bg_main"]};
    --bg-card: {COLORS["bg_card"]};
    --bg-alt: {COLORS["bg_alt"]};
    --bg-glass: {COLORS["bg_glass"]};
    --border-glass: {COLORS["border_glass"]};
    --accent: {COLORS["accent"]};
    --cyan: {COLORS["cyan"]};
    --purple: {COLORS["purple"]};
    --text-heading: {COLORS["text_heading"]};
    --text-muted: {COLORS["text_muted"]};
}}

html, body, [class*="css"], .stApp {{
    font-family: 'Inter', sans-serif;
    color: var(--text-heading);
    background: radial-gradient(circle at 15% 0%, #131c31 0%, #0B1220 45%, #090e19 100%) !important;
}}

section[data-testid="stSidebar"] {{
    background: linear-gradient(180deg, #0D1524 0%, #0B1220 100%) !important;
    border-right: 1px solid var(--border-glass);
}}

/* Kill the default white Streamlit header/toolbar bar */
header[data-testid="stHeader"] {{
    background: var(--bg-main) !important;
    box-shadow: none !important;
}}
div[data-testid="stDecoration"] {{ display: none !important; }}
div[data-testid="stToolbar"] {{ background: transparent !important; }}
#MainMenu {{ visibility: hidden; }}
footer {{ visibility: hidden; }}

/* Any bordered st.container(border=True) gets the same glass treatment */
[data-testid="stVerticalBlockBorderWrapper"] {{
    background: var(--bg-glass) !important;
    backdrop-filter: blur(14px);
    -webkit-backdrop-filter: blur(14px);
    border: 1px solid var(--border-glass) !important;
    border-radius: 20px !important;
}}

h1, h2, h3, h4, h5, h6 {{
    font-family: 'Space Grotesk', sans-serif;
    color: var(--text-heading);
    font-weight: 700;
}}

p, span, div, label {{ color: {COLORS["text_body"]}; }}

/* ---------------- Animations ---------------- */
@keyframes fadeInUp {{
    from {{ opacity: 0; transform: translateY(14px); }}
    to   {{ opacity: 1; transform: translateY(0); }}
}}
@keyframes glowPulse {{
    0%, 100% {{ box-shadow: 0 0 12px rgba(59,130,246,0.35), 0 0 0 rgba(139,92,246,0); }}
    50%      {{ box-shadow: 0 0 26px rgba(59,130,246,0.65), 0 0 14px rgba(139,92,246,0.35); }}
}}
@keyframes shimmer {{
    0%   {{ background-position: -400px 0; }}
    100% {{ background-position: 400px 0; }}
}}
@keyframes floatIcon {{
    0%, 100% {{ transform: translateY(0); }}
    50%      {{ transform: translateY(-3px); }}
}}
@keyframes borderFlow {{
    0%   {{ background-position: 0% 50%; }}
    50%  {{ background-position: 100% 50%; }}
    100% {{ background-position: 0% 50%; }}
}}
@keyframes dotBounce {{
    0%, 80%, 100% {{ transform: scale(0.6); opacity: 0.4; }}
    40%           {{ transform: scale(1); opacity: 1; }}
}}
@keyframes particleDrift {{
    0%   {{ transform: translate(0,0); opacity: .25; }}
    50%  {{ transform: translate(12px,-18px); opacity: .55; }}
    100% {{ transform: translate(0,0); opacity: .25; }}
}}

.fade-in {{ animation: fadeInUp 0.55s ease both; }}

/* ---------------- Hero ---------------- */
.hero-wrap {{
    position: relative;
    padding: 44px 40px;
    border-radius: 22px;
    margin-bottom: 26px;
    overflow: hidden;
    background: linear-gradient(135deg, rgba(59,130,246,0.16), rgba(139,92,246,0.14) 55%, rgba(6,182,212,0.14));
    border: 1px solid var(--border-glass);
    box-shadow: 0 20px 60px rgba(0,0,0,0.45);
    animation: fadeInUp 0.7s ease both;
}}
.hero-wrap::before {{
    content: "";
    position: absolute; inset: -40%;
    background: radial-gradient(circle at 30% 30%, rgba(59,130,246,0.25), transparent 55%),
                radial-gradient(circle at 80% 70%, rgba(139,92,246,0.25), transparent 55%);
    animation: particleDrift 9s ease-in-out infinite;
    pointer-events: none;
}}
.hero-eyebrow {{
    display: inline-block; font-family: 'JetBrains Mono', monospace; font-size: 13px;
    letter-spacing: 1.5px; color: var(--cyan); font-weight: 700; margin-bottom: 10px;
}}
.hero-title {{
    font-size: 42px; font-weight: 800; margin: 0; line-height: 1.1;
    background: {COLORS["grad_main"]};
    -webkit-background-clip: text; background-clip: text; color: transparent;
}}
.hero-sub {{
    font-size: 16px; color: var(--text-muted); margin: 14px 0 26px; max-width: 620px;
    font-weight: 500;
}}
.hero-pills {{ display: flex; gap: 10px; flex-wrap: wrap; margin-bottom: 4px; }}
.hero-pill {{
    font-size: 13px; font-weight: 700; padding: 6px 14px; border-radius: 999px;
    background: rgba(255,255,255,0.05); border: 1px solid var(--border-glass); color: var(--text-heading);
}}

/* ---------------- Glass Card ---------------- */
.glass-card {{
    background: var(--bg-glass);
    backdrop-filter: blur(14px);
    -webkit-backdrop-filter: blur(14px);
    border: 1px solid var(--border-glass);
    border-radius: 18px;
    padding: 22px;
    margin-bottom: 20px;
    box-shadow: 0 8px 32px rgba(0,0,0,0.35);
    transition: transform 0.25s ease, box-shadow 0.25s ease, border-color 0.25s ease;
    animation: fadeInUp 0.5s ease both;
}}
.glass-card:hover {{
    transform: translateY(-4px);
    border-color: rgba(59,130,246,0.45);
    box-shadow: 0 16px 42px rgba(59,130,246,0.22);
}}

/* ---------------- KPI Cards ---------------- */
.kpi-card {{
    position: relative;
    background: var(--bg-glass);
    backdrop-filter: blur(14px);
    border: 1px solid var(--border-glass);
    border-radius: 18px;
    padding: 20px 22px;
    overflow: hidden;
    transition: transform 0.25s ease, box-shadow 0.25s ease;
    animation: fadeInUp 0.6s ease both;
}}
.kpi-card:hover {{ transform: translateY(-5px) scale(1.01); box-shadow: 0 16px 40px rgba(0,0,0,0.5); }}
.kpi-card::after {{
    content: ""; position: absolute; top: -30%; right: -30%; width: 140px; height: 140px;
    background: radial-gradient(circle, var(--kpi-glow, rgba(59,130,246,0.35)), transparent 70%);
    pointer-events: none;
}}
.kpi-icon {{ font-size: 22px; margin-bottom: 6px; display: inline-block; animation: floatIcon 3s ease-in-out infinite; }}
.kpi-label {{ font-size: 12.5px; color: var(--text-muted); font-weight: 600; text-transform: uppercase; letter-spacing: 0.6px; }}
.kpi-value {{ font-size: 30px; font-weight: 800; color: var(--text-heading); margin: 4px 0 6px; font-family: 'Space Grotesk', sans-serif; }}
.kpi-delta-up   {{ color: {COLORS["green"]}; font-weight: 700; font-size: 13px; }}
.kpi-delta-down {{ color: {COLORS["red"]}; font-weight: 700; font-size: 13px; }}
.kpi-bar {{ height: 6px; border-radius: 4px; background: rgba(255,255,255,0.08); margin-top: 10px; overflow: hidden; }}
.kpi-bar-fill {{ height: 100%; border-radius: 4px; background: {COLORS["grad_main"]}; }}

/* ---------------- Badges ---------------- */
.agent-badge {{
    display: inline-block; padding: 4px 14px; border-radius: 999px; font-size: 12.5px; font-weight: 700;
    background: rgba(59,130,246,0.15); border: 1px solid rgba(59,130,246,0.4); color: #93C5FD;
    font-family: 'Space Grotesk', sans-serif;
}}
.pn-badge {{
    display: inline-block; padding: 4px 12px; border-radius: 8px; font-family: 'JetBrains Mono', monospace;
    font-weight: 700; font-size: 12.5px; border: 1px solid var(--border-glass); text-transform: uppercase;
}}

/* ---------------- Agent Status Cards ---------------- */
.agent-flow-card {{
    background: var(--bg-glass); border: 1px solid var(--border-glass); border-radius: 16px;
    padding: 16px 20px; margin-bottom: 12px; animation: fadeInUp 0.5s ease both;
    border-left: 3px solid var(--agent-color, var(--accent));
}}
.agent-flow-title {{ font-weight: 700; font-size: 15px; color: var(--text-heading); display:flex; align-items:center; gap:8px; }}
.agent-status-thinking {{ color: {COLORS["yellow"]}; font-size: 13px; font-weight: 600; }}
.agent-status-done {{ color: {COLORS["green"]}; font-size: 13px; font-weight: 600; }}
.thinking-dots span {{
    display: inline-block; width: 6px; height: 6px; margin-right: 3px; border-radius: 50%;
    background: {COLORS["yellow"]}; animation: dotBounce 1.2s infinite ease-in-out;
}}
.thinking-dots span:nth-child(2) {{ animation-delay: 0.15s; }}
.thinking-dots span:nth-child(3) {{ animation-delay: 0.3s; }}

/* ---------------- Alerts ---------------- */
.alert-card {{
    display: flex; align-items: center; justify-content: space-between; gap: 14px;
    background: var(--bg-glass); border: 1px solid var(--border-glass); border-left: 4px solid var(--alert-color, {COLORS["red"]});
    border-radius: 14px; padding: 14px 18px; margin-bottom: 10px; animation: fadeInUp 0.5s ease both;
    transition: transform 0.2s ease;
}}
.alert-card:hover {{ transform: translateX(3px); }}
.alert-title {{ font-weight: 700; font-size: 13.5px; }}
.alert-desc {{ font-size: 13px; color: var(--text-muted); margin-top: 2px; }}

/* ---------------- AI Recommendation Floating Panel ---------------- */
.ai-rec-panel {{
    background: linear-gradient(135deg, rgba(59,130,246,0.14), rgba(139,92,246,0.12));
    border: 1px solid rgba(139,92,246,0.35); border-radius: 20px; padding: 22px;
    box-shadow: 0 10px 40px rgba(139,92,246,0.18); animation: glowPulse 4s ease-in-out infinite;
}}
.ai-rec-title {{ font-size: 13px; font-weight: 700; color: var(--cyan); letter-spacing: 0.8px; text-transform: uppercase; }}
.ai-rec-body {{ font-size: 17px; font-weight: 700; margin: 8px 0 14px; }}
.ai-rec-metric {{ display:flex; justify-content: space-between; font-size: 13px; color: var(--text-muted); margin-bottom: 4px; }}
.ai-rec-metric b {{ color: var(--text-heading); }}

/* ---------------- Chat / Copilot ---------------- */
.copilot-hero {{
    text-align: center; padding: 10px 0 22px;
}}
.copilot-hero h2 {{ font-size: 26px; margin-bottom: 4px; }}
.chat-bubble-user {{
    background: {COLORS["grad_main"]}; color: white; border-radius: 16px 16px 4px 16px;
    padding: 14px 18px; margin: 8px 0; max-width: 78%; margin-left: auto; font-weight: 500;
    animation: fadeInUp 0.4s ease both; box-shadow: 0 6px 20px rgba(59,130,246,0.25);
}}
.chat-bubble-ai {{
    background: var(--bg-glass); border: 1px solid var(--border-glass); color: var(--text-heading);
    border-radius: 16px 16px 16px 4px; padding: 14px 18px; margin: 8px 0; max-width: 82%;
    animation: fadeInUp 0.4s ease both;
}}

/* ---------------- Buttons ---------------- */
div.stButton > button {{
    position: relative; overflow: hidden;
    background: {COLORS["grad_main"]} !important;
    background-size: 200% 200% !important;
    color: white !important;
    font-family: 'Space Grotesk', sans-serif !important;
    font-weight: 700 !important;
    border: none !important;
    border-radius: 12px !important;
    padding: 11px 24px !important;
    box-shadow: 0 6px 22px rgba(59,130,246,0.35) !important;
    transition: all 0.2s ease !important;
}}
div.stButton > button:hover {{
    transform: translateY(-2px) !important;
    box-shadow: 0 10px 30px rgba(139,92,246,0.45) !important;
    background-position: 100% 0 !important;
}}
div.stButton > button:active {{ transform: translateY(0) scale(0.98) !important; }}

/* Inputs — wrapper */
div[data-baseweb="input"] > div, div[data-baseweb="select"] > div, div[data-baseweb="textarea"] > div {{
    background: rgba(255,255,255,0.04) !important;
    border: 1px solid var(--border-glass) !important;
    border-radius: 12px !important;
    color: var(--text-heading) !important;
}}

/* Inputs — the actual editable element (this is what was invisible: light text
   was being rendered on the browser's default WHITE input background because
   only the wrapper div above was themed, not the <input>/<textarea> itself) */
div[data-baseweb="input"] input,
div[data-baseweb="textarea"] textarea,
input, textarea {{
    background: transparent !important;
    color: var(--text-heading) !important;
    -webkit-text-fill-color: var(--text-heading) !important;
    caret-color: var(--text-heading) !important;
}}
input::placeholder, textarea::placeholder {{
    color: var(--text-muted) !important;
    opacity: 1 !important;
    -webkit-text-fill-color: var(--text-muted) !important;
}}
/* Chrome/Edge autofill forces a white box + black text unless overridden like this */
input:-webkit-autofill, input:-webkit-autofill:hover, input:-webkit-autofill:focus {{
    -webkit-box-shadow: 0 0 0px 1000px rgba(255,255,255,0.04) inset !important;
    -webkit-text-fill-color: var(--text-heading) !important;
    caret-color: var(--text-heading) !important;
}}
/* Select / dropdown text + chevron icon (belt-and-braces on top of the
   native dark theme set in .streamlit/config.toml) */
div[data-baseweb="select"] * {{ color: var(--text-heading) !important; fill: var(--text-heading) !important; }}
div[data-baseweb="popover"] {{
    background: var(--bg-alt) !important;
    border: 1px solid var(--border-glass) !important;
    border-radius: 12px !important;
    box-shadow: 0 16px 40px rgba(0,0,0,0.5) !important;
    animation: dropdownIn 0.18s cubic-bezier(0.16, 1, 0.3, 1) both;
}}
@keyframes dropdownIn {{
    from {{ opacity: 0; transform: translateY(-6px) scale(0.97); }}
    to   {{ opacity: 1; transform: translateY(0) scale(1); }}
}}
ul[role="listbox"] {{ background: var(--bg-alt) !important; }}
li[role="option"] {{
    color: var(--text-heading) !important;
    background: transparent !important;
    transition: background 0.15s ease, transform 0.15s ease, padding-left 0.15s ease;
}}
li[role="option"]:hover {{
    background: rgba(59,130,246,0.16) !important;
    padding-left: 20px !important;
}}
li[aria-selected="true"] {{
    background: rgba(59,130,246,0.22) !important;
    color: var(--text-heading) !important;
}}

/* Dataframe / table — give the native dark-themed grid a matching glass frame
   instead of it looking like a plain rectangle dropped onto the card */
[data-testid="stDataFrame"], [data-testid="stTable"] {{
    border-radius: 16px !important;
    overflow: hidden !important;
    border: 1px solid var(--border-glass) !important;
    box-shadow: 0 8px 28px rgba(0,0,0,0.35) !important;
    animation: fadeInUp 0.5s ease both;
}}

/* Tabs */
button[data-baseweb="tab"] {{ font-family: 'Space Grotesk', sans-serif !important; font-weight: 700 !important; color: var(--text-muted) !important; }}
button[data-baseweb="tab"][aria-selected="true"] {{ color: var(--text-heading) !important; border-bottom: 3px solid var(--accent) !important; }}

/* ---------------- Sidebar Nav (native st.button, fully theme-controlled) ---------------- */
/* We build the sidebar nav out of real st.button() calls instead of the
   streamlit_option_menu component: that component renders inside its own
   iframe, which our page CSS cannot reach, and that's why it stayed white.
   Native buttons live in the same DOM as everything else, so this CSS
   applies fully. Selected item uses type="primary", others type="secondary". */
section[data-testid="stSidebar"] div.stButton > button {{
    background: transparent !important;
    box-shadow: none !important;
    color: var(--text-muted) !important;
    text-align: left !important;
    justify-content: flex-start !important;
    font-weight: 600 !important;
    font-family: 'Inter', sans-serif !important;
    border: 1px solid transparent !important;
    border-radius: 12px !important;
    padding: 10px 14px !important;
    margin-bottom: 4px !important;
    width: 100% !important;
    transition: all 0.2s ease !important;
}}
section[data-testid="stSidebar"] div.stButton > button:hover {{
    background: rgba(255,255,255,0.05) !important;
    color: var(--text-heading) !important;
    transform: translateX(4px) !important;
    box-shadow: none !important;
}}
section[data-testid="stSidebar"] div.stButton > button:active {{ transform: translateX(4px) scale(0.99) !important; }}
section[data-testid="stSidebar"] div.stButton > button[kind="primary"] {{
    background: linear-gradient(90deg, rgba(59,130,246,0.28), rgba(139,92,246,0.22)) !important;
    color: var(--text-heading) !important;
    border-left: 3px solid var(--accent) !important;
    box-shadow: 0 0 18px rgba(59,130,246,0.35) !important;
    animation: glowPulse 3.5s ease-in-out infinite;
}}
section[data-testid="stSidebar"] div.stButton > button[kind="primary"]:hover {{ transform: translateX(4px) !important; }}

/* Skeleton loader */
.skeleton {{
    height: 18px; border-radius: 8px; margin-bottom: 8px;
    background: linear-gradient(90deg, rgba(255,255,255,0.05) 25%, rgba(255,255,255,0.12) 37%, rgba(255,255,255,0.05) 63%);
    background-size: 400px 100%;
    animation: shimmer 1.4s ease-in-out infinite;
}}

/* Typing animation caret */
.typing-caret::after {{
    content: '▍'; animation: dotBounce 1s infinite; color: var(--accent);
}}

hr {{ border-color: var(--border-glass) !important; }}
</style>
"""


def inject_css():
    st.markdown(PREMIUM_CSS, unsafe_allow_html=True)


def apply_theme():
    inject_css()


def render_header(title, subtitle="", icon="⚡"):
    inject_css()
    st.markdown(f"""
    <div class="glass-card fade-in" style="display:flex;align-items:center;gap:16px;padding:20px 26px;">
        <div style="font-size:38px;line-height:1;">{icon}</div>
        <div>
            <h1 style="margin:0;font-size:24px;letter-spacing:-0.5px;">{title}</h1>
            <p style="margin:4px 0 0;color:{COLORS['text_muted']};font-size:14px;">{subtitle}</p>
        </div>
    </div>
    """, unsafe_allow_html=True)


def render_card(content, alt=False):
    st.markdown(f'<div class="glass-card">{content}</div>', unsafe_allow_html=True)


def risk_badge(text, level="Low"):
    color_map = {"Low": COLORS["green"], "Medium": COLORS["yellow"], "High": COLORS["red"], "Critical": COLORS["red"]}
    c = color_map.get(level, COLORS["cyan"])
    return f'<span class="pn-badge" style="background:rgba(255,255,255,0.06);color:{c};border-color:{c};">{text}</span>'


# ─────────────────────────────────────────────────────────────────────────
# Hero section (Home page)
# ─────────────────────────────────────────────────────────────────────────
def render_hero(cta_label="🚀 Launch AI Copilot", cta_key="hero_launch_cta"):
    inject_css()
    st.markdown(f"""
    <div class="hero-wrap">
        <div class="hero-eyebrow">⚡ FRANCHISEOPS AI</div>
        <div class="hero-title">Enterprise Multi-Agent<br>Intelligence Platform</div>
        <div class="hero-sub">Predict. Monitor. Optimize. Automate. — one AI copilot orchestrating
        workforce, outlet and inventory intelligence across your entire franchise network in real time.</div>
        <div class="hero-pills">
            <span class="hero-pill">🔮 Predict</span>
            <span class="hero-pill">📡 Monitor</span>
            <span class="hero-pill">⚙️ Optimize</span>
            <span class="hero-pill">🤖 Automate</span>
        </div>
    </div>
    """, unsafe_allow_html=True)
    return st.button(cta_label, key=cta_key, use_container_width=False)


# ─────────────────────────────────────────────────────────────────────────
# Animated KPI counter (pure HTML/CSS/JS component, no extra deps)
# ─────────────────────────────────────────────────────────────────────────
def render_kpi_row(kpis):
    """
    kpis: list of dicts:
      {label, value (numeric), prefix, suffix, icon, delta (str, optional),
       delta_up (bool), glow (hex/rgba, optional), decimals(int, optional), progress(0-100 optional)}
    """
    inject_css()
    cols = st.columns(len(kpis))
    for col, k in zip(cols, kpis):
        rid = f"kpi_{abs(hash(k['label']))}"
        delta_html = ""
        if k.get("delta"):
            cls = "kpi-delta-up" if k.get("delta_up", True) else "kpi-delta-down"
            arrow = "↑" if k.get("delta_up", True) else "↓"
            delta_html = f'<div class="{cls}">{arrow} {k["delta"]}</div>'
        bar_html = ""
        if k.get("progress") is not None:
            bar_html = f'''<div class="kpi-bar"><div class="kpi-bar-fill" style="width:{k["progress"]}%;"></div></div>'''
        glow = k.get("glow", "rgba(59,130,246,0.35)")
        with col:
            st.markdown(f"""
            <div class="kpi-card fade-in" style="--kpi-glow:{glow};">
                <div class="kpi-icon">{k.get('icon','📊')}</div>
                <div class="kpi-label">{k['label']}</div>
                <div class="kpi-value" id="{rid}">0</div>
                {delta_html}
                {bar_html}
            </div>
            <script>
            (function() {{
                const el = window.parent.document.getElementById("{rid}") || document.getElementById("{rid}");
                if (!el) return;
                const target = {k.get('value', 0)};
                const decimals = {k.get('decimals', 0)};
                const prefix = "{k.get('prefix','')}";
                const suffix = "{k.get('suffix','')}";
                let cur = 0;
                const steps = 40;
                const inc = target / steps;
                let i = 0;
                const timer = setInterval(function() {{
                    i++;
                    cur += inc;
                    if (i >= steps) {{ cur = target; clearInterval(timer); }}
                    el.textContent = prefix + cur.toFixed(decimals) + suffix;
                }}, 18);
            }})();
            </script>
            """, unsafe_allow_html=True)


# ─────────────────────────────────────────────────────────────────────────
# Agent flow visualization card (Thinking → Complete)
# ─────────────────────────────────────────────────────────────────────────
def render_agent_flow_card(name, status="done", color=None, icon="🤖"):
    """status: 'thinking' | 'done'"""
    color = color or COLORS["accent"]
    if status == "thinking":
        status_html = ('<span class="agent-status-thinking">Thinking'
                        '<span class="thinking-dots"><span></span><span></span><span></span></span></span>')
    else:
        status_html = '<span class="agent-status-done">✓ Complete</span>'
    st.markdown(f"""
    <div class="agent-flow-card" style="--agent-color:{color};">
        <div class="agent-flow-title">{icon} {name}</div>
        <div style="margin-top:6px;">{status_html}</div>
    </div>
    """, unsafe_allow_html=True)


# ─────────────────────────────────────────────────────────────────────────
# Alerts
# ─────────────────────────────────────────────────────────────────────────
def render_alert(title, desc, level="Critical", link_label="View →"):
    color_map = {"Critical": COLORS["red"], "Warning": COLORS["yellow"], "Info": COLORS["cyan"], "Success": COLORS["green"]}
    icon_map = {"Critical": "🔴", "Warning": "🟡", "Info": "🔵", "Success": "🟢"}
    c = color_map.get(level, COLORS["red"])
    icon = icon_map.get(level, "🔴")
    st.markdown(f"""
    <div class="alert-card" style="--alert-color:{c};">
        <div>
            <div class="alert-title">{icon} {level}</div>
            <div class="alert-desc">{desc if desc else title}</div>
        </div>
        <div style="color:{c};font-weight:700;font-size:13px;white-space:nowrap;">{link_label}</div>
    </div>
    """, unsafe_allow_html=True)


# ─────────────────────────────────────────────────────────────────────────
# Floating AI recommendation panel
# ─────────────────────────────────────────────────────────────────────────
def render_ai_recommendation(title, action, savings, confidence):
    st.markdown(f"""
    <div class="ai-rec-panel fade-in">
        <div class="ai-rec-title">✨ AI Recommendation</div>
        <div class="ai-rec-body">{action}</div>
        <div class="ai-rec-metric"><span>Expected Savings</span><b>{savings}</b></div>
        <div class="ai-rec-metric"><span>Confidence</span><b>{confidence}</b></div>
    </div>
    """, unsafe_allow_html=True)


def render_skeleton(lines=3):
    html = "".join(f'<div class="skeleton" style="width:{92 - i*8}%;"></div>' for i in range(lines))
    st.markdown(f'<div class="glass-card">{html}</div>', unsafe_allow_html=True)


# ─────────────────────────────────────────────────────────────────────────
# Password strength (Section 6 of the security spec)
# ─────────────────────────────────────────────────────────────────────────
def password_strength(pw):
    """Returns (level, label) — level in {'weak','average','good'}."""
    n = len(pw or "")
    if n < 5:
        return "weak", "🔴 Weak"
    elif n < 10:
        return "average", "🟡 Average"
    return "good", "🟢 Good"


def password_is_valid(pw):
    """Submission gate: anything under 5 chars is blocked outright."""
    return len(pw or "") >= 5


def render_password_strength(pw):
    """Live strength meter shown under a password field as the user types."""
    if not pw:
        return
    level, label = password_strength(pw)
    color = {"weak": COLORS["red"], "average": COLORS["yellow"], "good": COLORS["green"]}[level]
    note = {
        "weak": "Minimum 5 characters required.",
        "average": "Average strength — 10+ characters recommended for enterprise security.",
        "good": "Good password strength.",
    }[level]
    pct = min(100, int(len(pw) / 12 * 100))
    st.markdown(f'''
    <div class="fade-in" style="margin:-6px 0 14px;">
        <div style="display:flex;justify-content:space-between;font-size:12px;font-weight:700;color:{color};">
            <span>{label}</span><span>{len(pw)} chars</span>
        </div>
        <div class="kpi-bar" style="margin-top:4px;">
            <div class="kpi-bar-fill" style="width:{pct}%;background:{color};"></div>
        </div>
        <div style="font-size:11.5px;color:{COLORS["text_muted"]};margin-top:3px;">{note}</div>
    </div>''', unsafe_allow_html=True)

In [ ]:
%%writefile auth.py
"""
FranchiseOps AI - auth.py
Split-screen enterprise login (dark, blue-gradient, animated particles) +
SQLite auth with Login, Register (Enterprise roles), and Forgot Password
(Security Question OR Email OTP).

Security features:
  5.  Progressive account lockout (3rd/4th/5th consecutive failed attempts)
  5.1 OTP resend rate limiting (Gmail OTP cooldown)
  6.  Password strength policy (real-time checker + submission gate)
"""
import sqlite3, jwt, bcrypt, datetime, random, time, smtplib, streamlit as st
from email.mime.text import MIMEText

try:
    from config import DB_PATH, JWT_SECRET_KEY, EMAIL_ID, EMAIL_PASSWORD
    JWT_SECRET = JWT_SECRET_KEY
except (ImportError, AttributeError):
    from config import DB_PATH
    JWT_SECRET = "super-secret-franchiseops-key-2026"
    EMAIL_ID = EMAIL_PASSWORD = ""

from ui_theme import COLORS, inject_css, password_strength, password_is_valid, render_password_strength


def get_conn():
    return sqlite3.connect(DB_PATH, check_same_thread=False)


def hash_txt(t):
    return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()


def check_txt(t, h):
    try: return bcrypt.checkpw(t.encode(), h.encode()) if h else False
    except: return False


def make_jwt(email, username):
    return jwt.encode({"email": email, "username": username,
                        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=6)},
                       JWT_SECRET, algorithm="HS256")


def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None


@st.cache_resource
def init_auth():
    with get_conn() as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            security_question TEXT,
            security_answer_hash TEXT,
            role TEXT DEFAULT 'Franchisee',
            failed_attempts INTEGER DEFAULT 0,
            lock_until TIMESTAMP DEFAULT NULL,
            account_status TEXT DEFAULT 'active',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )""")
        for stmt in [
            "ALTER TABLE users ADD COLUMN security_question TEXT",
            "ALTER TABLE users ADD COLUMN security_answer_hash TEXT",
            "ALTER TABLE users ADD COLUMN failed_attempts INTEGER DEFAULT 0",
            "ALTER TABLE users ADD COLUMN lock_until TIMESTAMP DEFAULT NULL",
            "ALTER TABLE users ADD COLUMN account_status TEXT DEFAULT 'active'",
        ]:
            try: conn.execute(stmt)
            except Exception: pass

        if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
            conn.execute("""INSERT OR IGNORE INTO users
                         (username, email, password_hash, security_question, security_answer_hash,
                          role, account_status)
                         VALUES (?, ?, ?, ?, ?, ?, ?)""",
                         ("Administrator", "infosys@ai", hash_txt("admin@123"),
                          "What is your pet name?", hash_txt("admin"), "Admin", "active"))
        else:
            # Fix pre-existing accounts (persisted DBs) so the Admin sidebar/dashboard
            # actually appears — is_admin checks for role == 'Admin' exactly.
            conn.execute("UPDATE users SET role='Admin' WHERE email='infosys@ai' AND role!='Admin'")
        conn.commit()


# ─────────────────────────────────────────────────────────────────────────
# 5. Progressive account lockout
# ─────────────────────────────────────────────────────────────────────────
_LOCKOUT_RULES = {3: 300, 4: 900}  # consecutive failed attempts -> lock seconds

def _get_login_row(identifier):
    with get_conn() as conn:
        return conn.execute(
            "SELECT id, failed_attempts, lock_until, account_status, password_hash, "
            "username, email, role FROM users WHERE email=? OR username=?",
            (identifier, identifier)).fetchone()


def _register_failed_attempt(user_id, current_failed):
    new_failed = current_failed + 1
    if new_failed == 3:
        lock_until = datetime.datetime.utcnow() + datetime.timedelta(seconds=300)
        with get_conn() as conn:
            conn.execute("UPDATE users SET failed_attempts=?, lock_until=? WHERE id=?",
                         (new_failed, lock_until.isoformat(), user_id)); conn.commit()
        return "⏳ Account temporarily locked for 5 minutes due to 3 failed attempts."
    elif new_failed == 4:
        lock_until = datetime.datetime.utcnow() + datetime.timedelta(seconds=900)
        with get_conn() as conn:
            conn.execute("UPDATE users SET failed_attempts=?, lock_until=? WHERE id=?",
                         (new_failed, lock_until.isoformat(), user_id)); conn.commit()
        return "⏳ Account temporarily locked for 15 minutes due to 4 failed attempts."
    elif new_failed >= 5:
        with get_conn() as conn:
            conn.execute("UPDATE users SET failed_attempts=?, account_status='locked', lock_until=NULL WHERE id=?",
                         (new_failed, user_id)); conn.commit()
        return ("❌ Account permanently locked due to 5 failed attempts. Only the System "
                "Administrator can unlock this account via the Admin Dashboard.")
    else:
        with get_conn() as conn:
            conn.execute("UPDATE users SET failed_attempts=? WHERE id=?", (new_failed, user_id)); conn.commit()
        return f"❌ Invalid email/username or password. ({new_failed}/3 attempts before a temporary lock.)"


def _attempt_login(identifier, password):
    """Returns (ok: bool, message_or_None, user_row_or_None)."""
    row = _get_login_row(identifier)
    if not row:
        return False, "Invalid email/username or password.", None
    uid, failed, lock_until_str, status, pw_hash, uname, uemail, urole = row
    now = datetime.datetime.utcnow()

    if status == "locked":
        return False, ("❌ Account permanently locked due to 5 failed attempts. Only the System "
                       "Administrator can unlock this account via the Admin Dashboard."), None

    lock_until_dt = datetime.datetime.fromisoformat(lock_until_str) if lock_until_str else None
    if lock_until_dt and now < lock_until_dt:
        remaining = int((lock_until_dt - now).total_seconds())
        mins, secs = divmod(remaining, 60)
        return False, f"⏳ Account temporarily locked. Try again in {mins}m {secs}s.", None

    if check_txt(password, pw_hash):
        # Successful login — including the "lock has expired" case — resets the counters.
        with get_conn() as conn:
            conn.execute("UPDATE users SET failed_attempts=0, lock_until=NULL WHERE id=?", (uid,))
            conn.commit()
        return True, None, (uname, uemail, urole)

    return False, _register_failed_attempt(uid, failed), None


# ─────────────────────────────────────────────────────────────────────────
# 5.1 OTP resend rate limiting (Gmail OTP)
# ─────────────────────────────────────────────────────────────────────────
_OTP_COOLDOWNS = {1: 60, 2: 180, 3: 300}  # resend # -> cooldown seconds; 4th+ = 3600

def _send_otp_email(to_email, otp):
    """Returns (success: bool, debug_reason: str). debug_reason never contains
    the secret values themselves — safe to display in the UI."""
    if not EMAIL_ID:
        return False, "EMAIL_ID secret is empty — check Colab Secrets panel (🔑 icon) and make sure the 'Notebook access' toggle is ON for EMAIL_ID."
    if not EMAIL_PASSWORD:
        return False, "EMAIL_PASSWORD secret is empty — check Colab Secrets panel (🔑 icon) and make sure the 'Notebook access' toggle is ON for EMAIL_PASSWORD."
    try:
        msg = MIMEText(f"Your FranchiseOps AI verification code is: {otp}\n"
                       f"This code expires in 10 minutes. If you didn't request this, ignore this email.")
        msg["Subject"] = "FranchiseOps AI — Password Reset OTP"
        msg["From"] = EMAIL_ID
        msg["To"] = to_email
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=10) as server:
            server.login(EMAIL_ID, EMAIL_PASSWORD)
            server.send_message(msg)
        return True, ""
    except smtplib.SMTPAuthenticationError:
        return False, ("Gmail rejected the login. EMAIL_PASSWORD must be a 16-character App Password "
                       "(Google Account → Security → 2-Step Verification → App passwords) — "
                       "not your normal Gmail password. Also check for stray spaces if you pasted it.")
    except smtplib.SMTPException as e:
        print(f"OTP email send failed: {e}")
        return False, f"SMTP error: {type(e).__name__}: {e}"
    except Exception as e:
        print(f"OTP email send failed: {e}")
        return False, f"{type(e).__name__}: {e}"


def _register_otp_resend():
    cnt = st.session_state.get("otp_resend_count", 0) + 1
    st.session_state["otp_resend_count"] = cnt
    cooldown = _OTP_COOLDOWNS.get(cnt, 3600)
    st.session_state["otp_next_allowed"] = time.time() + cooldown
    if cooldown == 60:
        msg = "⏳ Please wait 60 seconds before requesting another OTP."
    elif cooldown == 180:
        msg = "⏳ Please wait 3 minutes before requesting another OTP."
    elif cooldown == 300:
        msg = "⏳ Please wait 5 minutes before requesting another OTP."
    else:
        msg = "⚠️ Too many OTP requests. Please wait 1 hour before trying again."
    return msg


def _otp_resend_allowed():
    now = time.time()
    nxt = st.session_state.get("otp_next_allowed", 0)
    return now >= nxt, max(0, int(nxt - now))


_LOGIN_CSS = f"""
<style>
.stApp {{ background: radial-gradient(circle at 20% 20%, #14213d 0%, #0B1220 55%, #060a12 100%) !important; }}
[data-testid="stSidebar"] {{ display:none; }}

.login-visual {{
    position: relative; border-radius: 24px; overflow: hidden; min-height: 480px;
    background: linear-gradient(160deg, rgba(59,130,246,0.22), rgba(139,92,246,0.22) 50%, rgba(6,182,212,0.2));
    border: 1px solid rgba(255,255,255,0.08);
    display:flex; flex-direction:column; justify-content:center; align-items:flex-start;
    padding: 46px 40px;
}}
.login-visual::before {{
    content:""; position:absolute; inset:0;
    background-image:
        radial-gradient(2px 2px at 20% 30%, rgba(255,255,255,0.5) 0, transparent 100%),
        radial-gradient(2px 2px at 70% 65%, rgba(255,255,255,0.4) 0, transparent 100%),
        radial-gradient(1.5px 1.5px at 40% 80%, rgba(255,255,255,0.35) 0, transparent 100%),
        radial-gradient(2px 2px at 85% 20%, rgba(255,255,255,0.4) 0, transparent 100%),
        radial-gradient(1.5px 1.5px at 55% 45%, rgba(255,255,255,0.3) 0, transparent 100%);
    animation: particleDrift 7s ease-in-out infinite;
}}
.login-visual .glow-orb {{
    position:absolute; width:260px; height:260px; border-radius:50%;
    background: radial-gradient(circle, rgba(59,130,246,0.45), transparent 70%);
    top:-60px; right:-60px; filter: blur(10px); animation: glowPulse 5s ease-in-out infinite;
}}
.login-badge {{
    font-family:'JetBrains Mono', monospace; font-size:12.5px; letter-spacing:1.5px; color:#67E8F9;
    font-weight:700; margin-bottom:14px; z-index:1;
}}
.login-visual h1 {{
    font-size:34px; font-weight:800; margin:0 0 14px; z-index:1; line-height:1.15;
    background: {COLORS["grad_main"]}; -webkit-background-clip:text; background-clip:text; color:transparent;
}}
.login-visual p {{ color:#CBD5E1; font-size:14.5px; max-width:380px; z-index:1; }}
.login-feature {{ display:flex; align-items:center; gap:10px; margin-top:14px; z-index:1; color:#E2E8F0; font-size:13.5px; font-weight:600; }}

/* The right-hand form now sits in a real st.container(border=True), which is
   themed globally (see [data-testid="stVerticalBlockBorderWrapper"] in
   ui_theme.py) — we just add a touch of inner padding + heading spacing here. */
[data-testid="stVerticalBlockBorderWrapper"] {{ padding: 26px 30px; min-height: 480px; }}
.login-form-wrap h2, [data-testid="stVerticalBlockBorderWrapper"] h2 {{ font-size: 26px; margin: 0 0 4px; }}
.sub {{ color:{COLORS["text_muted"]}; font-size:13.5px; margin-bottom:26px; }}
</style>
"""


def render_auth_portal():
    init_auth()
    inject_css()
    st.markdown(_LOGIN_CSS, unsafe_allow_html=True)
    if "token" not in st.session_state: st.session_state["token"] = None
    if "auth_tab" not in st.session_state: st.session_state["auth_tab"] = "Login"

    left, right = st.columns([1, 1.05], gap="large")

    with left:
        st.markdown("""
        <div class="login-visual">
            <div class="glow-orb"></div>
            <div class="login-badge">⚡ FRANCHISEOPS AI</div>
            <h1>Enterprise Multi-Agent<br>Intelligence Platform</h1>
            <p>One unified AI copilot orchestrating workforce, outlet and inventory
            intelligence — predicting risk before it happens.</p>
            <div class="login-feature">🤖 3 specialized AI agents working in sync</div>
            <div class="login-feature">📡 Real-time monitoring across every outlet</div>
            <div class="login-feature">🔐 Enterprise-grade role-based access</div>
        </div>
        """, unsafe_allow_html=True)

    with right:
        # st.container(border=True) actually nests its children (unlike the raw
        # <div> markup wrap we used to use), and the global CSS themes every
        # bordered container the same glassy way.
        form_box = st.container(border=True)
        tab1, tab2, tab3 = form_box.tabs(["🔐 Sign In", "📝 Register", "🔑 Reset Password"])

        # ── Sign In ──────────────────────────────────────────────────────
        with tab1:
            st.markdown('<h2>Welcome back</h2><div class="sub">Sign in to your FranchiseOps AI account</div>', unsafe_allow_html=True)
            login_email = st.text_input("Email / Username", key="l_email", placeholder="infosys@ai")
            login_pw = st.text_input("Password", type="password", key="l_pw", placeholder="••••••••")
            if st.button("🚀 Sign In to Portal", key="btn_login", use_container_width=True):
                ok, msg, user = _attempt_login(login_email, login_pw)
                if ok:
                    uname, uemail, urole = user
                    st.session_state["token"] = make_jwt(uemail, uname)
                    st.session_state["username"] = uname
                    st.session_state["role"] = urole
                    st.success(f"Welcome back, {uname} [{urole}]!")
                    st.rerun()
                else:
                    st.error(msg)

        # ── Register ─────────────────────────────────────────────────────
        with tab2:
            st.markdown('<h2>Create account</h2><div class="sub">Register a new enterprise franchise account</div>', unsafe_allow_html=True)
            r_user = st.text_input("Username", key="r_u")
            r_email = st.text_input("Email Address", key="r_e")
            r_pw = st.text_input("Create Password", type="password", key="r_p")
            render_password_strength(r_pw)
            r_role = st.selectbox("Select Enterprise Role",
                                   ["Franchise Owner", "Regional Operations Manager", "Store Manager", "Supply Chain Analyst"],
                                   key="r_role")
            r_q = st.selectbox("Security Question",
                                ["What is your pet name?", "What city were you born in?", "What is your favorite school teacher's name?"],
                                key="r_q")
            r_a = st.text_input("Security Answer", key="r_a")
            if st.button("✨ Create Account", key="btn_reg", use_container_width=True):
                if not (r_user and r_email and r_pw and r_a):
                    st.warning("Please fill out all fields.")
                elif not password_is_valid(r_pw):
                    st.warning("🔴 Password too weak (minimum 5 characters required).")
                else:
                    try:
                        with get_conn() as conn:
                            conn.execute("""INSERT INTO users
                                (username, email, password_hash, security_question, security_answer_hash, role)
                                VALUES (?, ?, ?, ?, ?, ?)""",
                                (r_user, r_email, hash_txt(r_pw), r_q, hash_txt(r_a.lower().strip()), r_role))
                            conn.commit()
                        level, _ = password_strength(r_pw)
                        note = ("🟡 Average strength (10+ characters recommended for enterprise security)."
                               if level == "average" else "🟢 Good password strength — account secured with bcrypt hashing.")
                        st.success(f"Account registered with role [{r_role}]! {note} Switch to Sign In tab.")
                    except Exception:
                        st.error("Registration failed: Email or username may already exist.")

        # ── Reset Password ──────────────────────────────────────────────
        with tab3:
            st.markdown('<h2>Reset password</h2><div class="sub">Verify your identity to reset your password</div>', unsafe_allow_html=True)
            reset_method = st.radio("Reset method", ["🔑 Security Question", "📧 Email OTP"],
                                    key="reset_method", horizontal=True, label_visibility="collapsed")

            # -- Security Question flow (unchanged) --
            if reset_method == "🔑 Security Question":
                f_email = st.text_input("Registered Email", key="f_e")
                if st.button("Verify Email & Fetch Question", key="btn_f1", use_container_width=True):
                    with get_conn() as conn:
                        u = conn.execute("SELECT security_question FROM users WHERE email=?", (f_email,)).fetchone()
                    if u:
                        st.session_state["reset_email"] = f_email
                        st.session_state["reset_q"] = u[0]
                        st.rerun()
                    else:
                        st.error("Email not found.")

                if st.session_state.get("reset_email"):
                    st.info(f"Security Question: **{st.session_state.get('reset_q')}**")
                    ans_try = st.text_input("Enter Answer", key="f_ans")
                    new_pw = st.text_input("New Password", type="password", key="f_npw")
                    render_password_strength(new_pw)
                    if st.button("Confirm Password Reset", key="btn_f2", use_container_width=True):
                        with get_conn() as conn:
                            u_hash = conn.execute("SELECT security_answer_hash FROM users WHERE email=?",
                                                  (st.session_state["reset_email"],)).fetchone()
                        if not (u_hash and check_txt(ans_try.lower().strip(), u_hash[0])):
                            st.error("Incorrect security answer.")
                        elif not password_is_valid(new_pw):
                            st.warning("🔴 Password too weak (minimum 5 characters required).")
                        else:
                            with get_conn() as conn:
                                conn.execute("UPDATE users SET password_hash=? WHERE email=?",
                                            (hash_txt(new_pw), st.session_state["reset_email"]))
                                conn.commit()
                            st.success("Password reset successfully! Please sign in.")
                            st.session_state["reset_email"] = None

            # -- Email OTP flow --
            else:
                st.markdown(f'<h4 style="margin:0 0 10px;">📧 Email OTP Verification</h4>', unsafe_allow_html=True)
                otp_email = st.text_input("Registered Email", key="otp_email")
                c1, c2 = st.columns(2)
                with c1:
                    send_clicked = st.button("📨 Send OTP", key="btn_send_otp", use_container_width=True)
                with c2:
                    resend_clicked = st.button("🔁 Resend OTP", key="btn_resend_otp", use_container_width=True)

                if send_clicked or resend_clicked:
                    is_first_send = send_clicked and "otp_code" not in st.session_state
                    allowed, wait_secs = _otp_resend_allowed()
                    if is_first_send or allowed:
                        with get_conn() as conn:
                            u = conn.execute("SELECT id FROM users WHERE email=?", (otp_email,)).fetchone()
                        if not u:
                            st.error("Email not found.")
                        else:
                            otp = f"{random.randint(0, 999999):06d}"
                            st.session_state["otp_code"] = otp
                            st.session_state["otp_email_target"] = otp_email
                            st.session_state["otp_expires"] = time.time() + 600
                            sent_ok, send_err = _send_otp_email(otp_email, otp)
                            if resend_clicked:
                                st.info(_register_otp_resend())
                            if sent_ok:
                                st.success(f"OTP sent to {otp_email}.")
                            else:
                                st.warning(f"Email delivery isn't configured for this deployment — "
                                          f"for testing, your OTP is: **{otp}**")
                                st.caption(f"🔍 Debug reason: {send_err}")
                    else:
                        mins, secs = divmod(wait_secs, 60)
                        st.warning(f"⏳ Please wait {mins}m {secs}s before requesting another OTP.")

                if st.session_state.get("otp_code"):
                    entered_otp = st.text_input("Enter 6-digit OTP", key="otp_entered", max_chars=6)
                    new_pw_otp = st.text_input("New Password", type="password", key="otp_new_pw")
                    render_password_strength(new_pw_otp)
                    if st.button("Confirm Password Reset", key="btn_otp_confirm", use_container_width=True):
                        if time.time() > st.session_state.get("otp_expires", 0):
                            st.error("OTP expired. Please request a new one.")
                        elif entered_otp != st.session_state.get("otp_code"):
                            st.error("Incorrect OTP.")
                        elif not password_is_valid(new_pw_otp):
                            st.warning("🔴 Password too weak (minimum 5 characters required).")
                        else:
                            with get_conn() as conn:
                                conn.execute("UPDATE users SET password_hash=? WHERE email=?",
                                            (hash_txt(new_pw_otp), st.session_state["otp_email_target"]))
                                conn.commit()
                            st.success("Password reset successfully! Please sign in.")
                            for k in ["otp_code", "otp_email_target", "otp_expires",
                                     "otp_resend_count", "otp_next_allowed"]:
                                st.session_state.pop(k, None)

In [ ]:
%%writefile db.py
import sqlite3
from config import DB_PATH

def get_conn():
    return sqlite3.connect(DB_PATH, check_same_thread=False)

def init_db():
    with get_conn() as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS outlets (
            outlet_id TEXT PRIMARY KEY, outlet_name TEXT, city TEXT,
            monthly_revenue REAL, monthly_costs REAL, staff_headcount INTEGER,
            avg_overtime_hours REAL, customer_satisfaction REAL,
            tier_cluster TEXT, attrition_risk_level TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS staff (
            staff_id TEXT PRIMARY KEY, outlet_id TEXT, employee_name TEXT,
            role TEXT, monthly_salary REAL, weekly_overtime_hrs REAL,
            job_satisfaction INTEGER, employee_age INTEGER, tenure_years REAL,
            work_life_balance INTEGER, predicted_attrition_prob REAL,
            intervention_status TEXT DEFAULT 'Active')""")
        conn.execute("""CREATE TABLE IF NOT EXISTS inventory_records (
            record_id INTEGER PRIMARY KEY AUTOINCREMENT, outlet_id TEXT,
            sku_name TEXT, current_stock INTEGER, weekly_demand INTEGER,
            reorder_threshold INTEGER, stockout_risk_prob REAL,
            last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS merged_datasets (
            id INTEGER PRIMARY KEY AUTOINCREMENT, agent_target TEXT, dataset_source TEXT,
            outlet_id TEXT, employee_age INTEGER, overtime_hours REAL,
            job_satisfaction INTEGER, attrition_target INTEGER, monthly_sales_usd REAL,
            operating_cost_usd REAL, tier_cluster_label INTEGER, sku_demand INTEGER,
            weather_impact_factor REAL, stockout_target INTEGER,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE,
            email TEXT UNIQUE, password_hash TEXT,
            security_question TEXT, security_answer_hash TEXT,
            role TEXT DEFAULT 'User',
            failed_attempts INTEGER DEFAULT 0,
            lock_until TIMESTAMP DEFAULT NULL,
            account_status TEXT DEFAULT 'active',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        try: conn.execute("ALTER TABLE users ADD COLUMN security_question TEXT")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN security_answer_hash TEXT")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN failed_attempts INTEGER DEFAULT 0")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN lock_until TIMESTAMP DEFAULT NULL")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN account_status TEXT DEFAULT 'active'")
        except Exception: pass
        conn.execute("""CREATE TABLE IF NOT EXISTS ml_models (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            agent_name TEXT, model_name TEXT, r2_score REAL,
            rmse REAL, accuracy REAL, training_rows INTEGER,
            file_path TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS notifications (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            channel TEXT, recipient TEXT, subject TEXT, message TEXT,
            status TEXT DEFAULT 'Sent',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL, role TEXT NOT NULL, content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.commit()

def save_ml_metrics(agent_name, model_name, r2, rmse, acc, rows, path):
    with get_conn() as conn:
        conn.execute("INSERT INTO ml_models "
                     "(agent_name,model_name,r2_score,rmse,accuracy,training_rows,file_path) "
                     "VALUES (?,?,?,?,?,?,?)",
                     (agent_name, model_name, r2, rmse, acc, rows, path))
        conn.commit()

def load_chat_history(username, conn_fn=None, limit=60):
    fn = conn_fn or get_conn
    with fn() as conn:
        rows = conn.execute(
            "SELECT role,content FROM chat_history WHERE username=? "
            "ORDER BY id DESC LIMIT ?", (username, limit)).fetchall()
    return [{"role":r[0],"content":r[1]} for r in reversed(rows)]

def save_chat_message(username, role, content, conn_fn=None):
    fn = conn_fn or get_conn
    with fn() as conn:
        conn.execute("INSERT INTO chat_history (username,role,content) VALUES (?,?,?)",
                     (username, role, content))
        conn.commit()

def clear_chat_history(username, conn_fn=None):
    fn = conn_fn or get_conn
    with fn() as conn:
        conn.execute("DELETE FROM chat_history WHERE username=?", (username,))
        conn.commit()

In [ ]:
%%writefile weather_context.py
"""
weather_context.py for FranchiseOps AI
Simulates local Indian city weather disruptions and logistics delays across franchise outlets.
"""
import random

CITY_WEATHER_REPORTS = {
    "Mumbai (MH)": {"status": "Heavy Monsoon Rain & Waterlogging", "temp_c": 28, "demand_impact_pct": -18.0, "supply_delay_days": 2, "attrition_stress": "High"},
    "Bengaluru (KA)": {"status": "Pleasant / Light Showers", "temp_c": 24, "demand_impact_pct": 12.0, "supply_delay_days": 0, "attrition_stress": "Normal"},
    "Delhi NCR (DL)": {"status": "Intense Summer Heatwave & Smog", "temp_c": 42, "demand_impact_pct": 15.0, "supply_delay_days": 1, "attrition_stress": "High"},
    "Hyderabad (TG)": {"status": "Clear & Warm", "temp_c": 33, "demand_impact_pct": 8.0, "supply_delay_days": 0, "attrition_stress": "Normal"},
    "Chennai (TN)": {"status": "Humid & Coastal Showers", "temp_c": 35, "demand_impact_pct": -5.0, "supply_delay_days": 1, "attrition_stress": "Medium"},
    "Pune (MH)": {"status": "Cloudy & Breezy", "temp_c": 26, "demand_impact_pct": 10.0, "supply_delay_days": 0, "attrition_stress": "Normal"},
    "Ahmedabad (GJ)": {"status": "Dry & High Heat", "temp_c": 40, "demand_impact_pct": -8.0, "supply_delay_days": 1, "attrition_stress": "Medium"},
    "Kolkata (WB)": {"status": "Thunderstorms & High Humidity", "temp_c": 32, "demand_impact_pct": -12.0, "supply_delay_days": 2, "attrition_stress": "High"}
}

def get_city_weather(city_name):
    for k, v in CITY_WEATHER_REPORTS.items():
        if k.lower() in city_name.lower() or city_name.lower() in k.lower():
            return {"city": k, **v}
    return {"city": city_name, "status": "Fair Weather Conditions", "temp_c": 30, "demand_impact_pct": 0.0, "supply_delay_days": 0, "attrition_stress": "Normal"}

def get_weather_report(port_name):
    return {"port": port_name, "status": "Normal Marine Conditions", "temp_c": 25, "wind_kt": 15, "delay_penalty_multiplier": 1.00}


In [ ]:
%%writefile notifications.py
"""
FranchiseOps AI - notifications.py
Multi-channel alert center simulating SMS, Email, and In-App notifications stored in SQLite.
"""
from db import get_conn

def send_alert(channel, recipient, subject, message):
    with get_conn() as conn:
        conn.execute("INSERT INTO notifications (channel, recipient, subject, message, status) VALUES (?, ?, ?, ?, ?)",
                     (channel, recipient, subject, message, "Delivered"))
        conn.commit()
    print(f"[{channel.upper()}] To: {recipient} | Subject: {subject} | Status: Delivered")

def get_recent_alerts(limit=15):
    with get_conn() as conn:
        return conn.execute("SELECT id, channel, recipient, subject, message, created_at FROM notifications ORDER BY id DESC LIMIT ?", (limit,)).fetchall()


In [ ]:
%%writefile seed_data.py
"""
FranchiseOps AI - seed_data.py
Pre-seeds the database with realistic outlets, staff members, shift logs, and inventory benchmarks.
"""
from db import get_conn, init_db
from notifications import send_alert

def seed_all():
    init_db()
    with get_conn() as conn:
        # Seed Outlets
        if not conn.execute("SELECT count(*) FROM outlets").fetchone()[0]:
            outlets = [
                ("OUT-101", "Mumbai Flagship Store", "Mumbai (MH)", 145000, 112000, 24, 18.5, 4.2, "Tier 3 (At-Risk)", "High Attrition"),
                ("OUT-102", "Bengaluru Tech Hub Cafe", "Bengaluru (KA)", 285000, 165000, 32, 4.2, 4.8, "Tier 1 (Apex)", "Low Attrition"),
                ("OUT-103", "Delhi NCR Metro Express", "Delhi NCR (DL)", 210000, 155000, 28, 14.0, 4.5, "Tier 2 (Stable)", "Moderate Attrition"),
                ("OUT-104", "Hyderabad Central Hub", "Hyderabad (TG)", 125000, 118000, 18, 22.0, 3.8, "Tier 3 (At-Risk)", "Critical Attrition"),
                ("OUT-105", "Chennai Coastal Kiosk", "Chennai (TN)", 195000, 138000, 26, 6.5, 4.7, "Tier 1 (Apex)", "Low Attrition"),
                ("OUT-106", "Pune IT Park Outlet", "Pune (MH)", 172000, 129000, 22, 9.8, 4.4, "Tier 2 (Stable)", "Low Attrition"),
            ]
            conn.executemany("INSERT INTO outlets VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, CURRENT_TIMESTAMP)", outlets)

        # Seed Staff
        if not conn.execute("SELECT count(*) FROM staff").fetchone()[0]:
            staff = [
                ("ST-5001", "OUT-101", "Marcus Vance", "Shift Supervisor", 3920.0, 21.0, 2, 32, 4.5, 2, 0.82, "Retention Bonus Offered"),
                ("ST-5002", "OUT-101", "Elena Rostova", "Barista / Cashier", 2880.0, 19.5, 2, 26, 2.0, 2, 0.79, "Schedule Adjusted"),
                ("ST-5003", "OUT-102", "David Chen", "Store Manager", 5120.0, 3.5, 5, 41, 8.5, 4, 0.12, "Stable"),
                ("ST-5004", "OUT-104", "Samantha Diaz", "Kitchen Lead", 3360.0, 24.5, 1, 29, 3.0, 1, 0.89, "Immediate Review Required"),
                ("ST-5005", "OUT-105", "James Wilson", "Team Lead", 4000.0, 5.0, 4, 36, 6.0, 3, 0.18, "Stable"),
            ]
            conn.executemany("INSERT INTO staff (staff_id, outlet_id, employee_name, role, monthly_salary, weekly_overtime_hrs, job_satisfaction, employee_age, tenure_years, work_life_balance, predicted_attrition_prob, intervention_status) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", staff)

        # Seed Inventory
        if not conn.execute("SELECT count(*) FROM inventory_records").fetchone()[0]:
            inventory = [
                ("OUT-101", "Premium Coffee Beans (Kg)", 140, 320, 180, 0.84),
                ("OUT-101", "Organic Milk Syrups (L)", 85, 190, 100, 0.78),
                ("OUT-102", "Premium Coffee Beans (Kg)", 580, 450, 250, 0.12),
                ("OUT-104", "Eco-Packaging Cups (Box)", 40, 210, 150, 0.91),
                ("OUT-105", "Artisan Tea Blends (Kg)", 310, 220, 140, 0.15),
            ]
            conn.executemany("INSERT INTO inventory_records (outlet_id, sku_name, current_stock, weekly_demand, reorder_threshold, stockout_risk_prob) VALUES (?, ?, ?, ?, ?, ?)", inventory)
            conn.commit()

    send_alert("Email", "franchisee@franchiseops.ai", "Franchise Operations Initialized", "Database seeded with 6 regional outlets, staff logs, and inventory benchmarks.")
    print("✅ Database pre-seeded successfully.")


In [ ]:
%%writefile admin_dash.py
"""admin_dash.py — Shared Admin Dashboard renderer for FreightQuote & FranchiseOps AI"""
import subprocess, datetime
import streamlit as st
import pandas as pd
import plotly.express as px
from db import get_conn
from notifications import get_recent_alerts
from ui_theme import render_card, COLORS, password_is_valid, render_password_strength
from auth import hash_txt

_APP_START = datetime.datetime.now()


def _smi(query):
    try:
        r = subprocess.run(
            ["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=3)
        return r.stdout.strip()
    except Exception:
        return "N/A"


def render_admin_dashboard(project="freight"):
    render_card('<h3 style="margin:0;">🛡️ Admin Dashboard — System Intelligence</h3>')

    # ── 1. System Health ─────────────────────────────────────────────────────
    st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:16px 0 8px;">⚙️ System Health</h4>',
                unsafe_allow_html=True)
    gpu_mem  = _smi("memory.used")
    gpu_tot  = _smi("memory.total")
    gpu_util = _smi("utilization.gpu")
    uptime   = str(datetime.datetime.now() - _APP_START).split(".")[0]
    h1, h2, h3, h4 = st.columns(4)
    for col, icon, label, val in [
        (h1, "🖥️", "GPU VRAM Used",  f"{gpu_mem} / {gpu_tot} MB"),
        (h2, "⚡", "GPU Utilization", f"{gpu_util}%"),
        (h3, "🕒", "App Uptime",      uptime),
        (h4, "✅", "LLM Status",      "Active" if gpu_mem != "N/A" else "Standby"),
    ]:
        col.markdown(
            f'<div class="pn-card" style="text-align:center;padding:14px;">'
            f'<div style="font-size:26px;">{icon}</div>'
            f'<h3 style="margin:6px 0 2px;font-size:1.1rem;">{val}</h3>'
            f'<p style="margin:0;color:{COLORS["text_muted"]};font-size:12px;">{label}</p>'
            f'</div>', unsafe_allow_html=True)

    st.markdown("---")

    # ── 2. User Management ───────────────────────────────────────────────────
    st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">👥 User Management</h4>',
                unsafe_allow_html=True)

    # -- Add User (spec section 9: Add User Portal) --
    with st.expander("➕ Add New User", expanded=False):
        with st.form("add_user_form", clear_on_submit=True):
            fc1, fc2 = st.columns(2)
            with fc1:
                au_user = st.text_input("Username")
                au_email = st.text_input("Email")
            with fc2:
                au_pw = st.text_input("Initial Password", type="password")
                au_role = st.selectbox("Role", ["Admin", "Franchise Owner", "Regional Operations Manager",
                                                "Store Manager", "Supply Chain Analyst"])
            submitted = st.form_submit_button("Create User", use_container_width=True)
            if submitted:
                if not (au_user and au_email and au_pw):
                    st.warning("Please fill out all fields.")
                elif not password_is_valid(au_pw):
                    st.warning("🔴 Password too weak (minimum 5 characters required).")
                else:
                    try:
                        with get_conn() as conn:
                            conn.execute(
                                "INSERT INTO users (username, email, password_hash, role, account_status) "
                                "VALUES (?, ?, ?, ?, 'active')",
                                (au_user, au_email, hash_txt(au_pw), au_role))
                            conn.commit()
                        st.success(f"✅ User {au_user} created with role [{au_role}].")
                        st.rerun()
                    except Exception:
                        st.error("Username or email already exists.")

    with get_conn() as conn:
        try:
            users_df = pd.read_sql(
                "SELECT id, username, role, email, failed_attempts, account_status, created_at "
                "FROM users ORDER BY id DESC", conn)
        except Exception:
            users_df = pd.DataFrame(columns=["id", "username", "role", "email",
                                             "failed_attempts", "account_status", "created_at"])

    if users_df.empty:
        st.info("No users registered yet.")
    else:
        for _, row in users_df.iterrows():
            uc1, uc2, uc3, uc4, uc5 = st.columns([2, 1.6, 1.4, 1, 1])
            uc1.markdown(f"**{row['username']}**<br>"
                        f"<span style='color:{COLORS['text_muted']};font-size:11.5px;'>{row['email']}</span>",
                        unsafe_allow_html=True)
            uc2.markdown(f'<span style="color:#60A5FA;font-weight:600;">[{row["role"]}]</span>',
                        unsafe_allow_html=True)

            is_locked = (row.get("account_status") == "locked") or ((row.get("failed_attempts") or 0) >= 3)
            status_html = (f'<span style="color:{COLORS["red"]};font-weight:700;">🔒 Locked</span>' if is_locked
                          else f'<span style="color:{COLORS["green"]};font-weight:700;">🟢 Active</span>')
            uc3.markdown(status_html, unsafe_allow_html=True)

            with uc4:
                if is_locked:
                    if st.button("🔓", key=f"unlock_user_{row['id']}", help=f"Unlock {row['username']}"):
                        with get_conn() as c:
                            c.execute("UPDATE users SET failed_attempts=0, lock_until=NULL, "
                                     "account_status='active' WHERE id=?", (row["id"],))
                            c.commit()
                        st.success(f"✅ {row['username']} account unlocked successfully.")
                        st.rerun()
            with uc5:
                if st.button("🗑️", key=f"del_user_{row['id']}", help=f"Delete {row['username']}"):
                    with get_conn() as c:
                        c.execute("DELETE FROM users WHERE id=?", (row["id"],))
                        c.commit()
                    st.success(f"Deleted {row['username']}")
                    st.rerun()

    st.markdown("---")

    # ── 3. LLM Activity Monitor ──────────────────────────────────────────────
    st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">🤖 LLM Activity Monitor</h4>',
                unsafe_allow_html=True)
    with get_conn() as conn:
        try:
            chat_df = pd.read_sql(
                "SELECT username, count(*) as queries FROM chat_history "
                "WHERE role='user' GROUP BY username ORDER BY queries DESC", conn)
            total_q = int(chat_df["queries"].sum()) if not chat_df.empty else 0
        except Exception:
            chat_df = pd.DataFrame(columns=["username","queries"])
            total_q = 0

    mc1, mc2 = st.columns([1, 1.6])
    with mc1:
        st.metric("Total Copilot Queries", total_q)
        st.dataframe(chat_df, use_container_width=True, hide_index=True)
    with mc2:
        if not chat_df.empty:
            fig = px.pie(chat_df, names="username", values="queries",
                         title="Queries per User", hole=0.4,
                         color_discrete_sequence=px.colors.sequential.Teal)
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                              height=250, margin=dict(l=10,r=10,t=40,b=10))
            st.plotly_chart(fig, use_container_width=True)

    st.markdown("---")

    # ── 4. ML Model Audit ────────────────────────────────────────────────────
    st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">📈 ML Model Audit</h4>',
                unsafe_allow_html=True)
    with get_conn() as conn:
        try:
            ml_df = pd.read_sql(
                "SELECT agent_name, model_name, r2_score, accuracy, "
                "training_rows, created_at FROM ml_models ORDER BY id DESC", conn)
        except Exception:
            ml_df = pd.DataFrame()
    if ml_df.empty:
        st.info("No model training records found. Run retraining from Analytics tab.")
    else:
        st.dataframe(ml_df, use_container_width=True, hide_index=True)

    st.markdown("---")

    # ── 5. Live Alert Log ────────────────────────────────────────────────────
    st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">🔔 Live Alert Log</h4>',
                unsafe_allow_html=True)
    filt = st.selectbox("Filter by type", ["All","In-App","Email","SMS"], key="admin_alert_filt")
    alerts = get_recent_alerts(50)
    for a in alerts:
        if filt != "All" and a[1].lower() != filt.lower():
            continue
        badge = {"email":"#ffd803","sms":"#f87171","in-app":"#34d399"}.get(a[1].lower(),"#bae8e8")
        st.markdown(
            f'<div style="border-left:4px solid {badge};padding:4px 10px;margin:3px 0;'
            f'font-size:13px;"><b>[{a[1].upper()}]</b> {a[3]} '
            f'<span style="color:{COLORS["text_muted"]};float:right;">{a[4]}</span></div>',
            unsafe_allow_html=True)

In [ ]:
%%writefile agent2_franchise.py
"""
agent2_franchise.py — Enriched Agent 2: Outlet Territory Clustering & City Weather
New features: City demand surge chart, revenue vs weather scatter, AI territory advisory.
Extended Indian cities + global franchise locations.
"""
import pandas as pd
import streamlit as st
import plotly.express as px
from ui_theme import render_card, COLORS
from db import get_conn
from weather_context import get_city_weather
from llm_engine import orchestrate_3_agents_query

# ── Full outlet / city list (heavy India coverage) ───────────────────────────
INDIA_CITIES = [
    "Mumbai (MH)", "Delhi (DL)", "Bengaluru (KA)", "Hyderabad (TS)",
    "Chennai (TN)", "Pune (MH)", "Kolkata (WB)", "Ahmedabad (GJ)",
    "Jaipur (RJ)", "Surat (GJ)", "Lucknow (UP)", "Chandigarh (PB)",
    "Bhopal (MP)", "Indore (MP)", "Nagpur (MH)", "Coimbatore (TN)",
    "Kochi (KL)", "Visakhapatnam (AP)", "Patna (BR)", "Ranchi (JH)",
]
GLOBAL_CITIES = [
    "Chicago (IL)", "Los Angeles (CA)", "New York (NY)", "Houston (TX)",
    "London (UK)", "Dubai (AE)", "Singapore (SG)",
]
ALL_CITIES = INDIA_CITIES + GLOBAL_CITIES


def render_agent2_franchise(agent2_c, agent2_r, username, db_stats, a1_ctx, a3_ctx,
                             send_alert, confidence_band):
    render_card('<h3 style="margin:0;">🏬 Agent 2: Outlet Territory Clustering</h3>')

    with get_conn() as conn:
        try:
            out_df = pd.read_sql("SELECT * FROM outlets", conn)
        except Exception:
            out_df = pd.DataFrame()

    c1, c2 = st.columns([1.3, 1])
    with c1:
        if not out_df.empty:
            st.dataframe(
                out_df[["outlet_id", "outlet_name", "city",
                        "monthly_revenue", "monthly_costs", "tier_cluster"]],
                use_container_width=True, hide_index=True)
            fig = px.scatter(
                out_df, x="monthly_costs", y="monthly_revenue",
                color="tier_cluster", size="staff_headcount",
                hover_name="outlet_name",
                title="Revenue vs Cost Clustering",
                color_discrete_sequence=["#34d399", "#ffd803", "#f87171"])
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                              height=280, margin=dict(l=10, r=10, t=40, b=10))
            st.plotly_chart(fig, use_container_width=True)

    with c2:
        render_card('<h4 style="margin:0 0 10px;">Simulate New Outlet</h4>')
        city_sel = st.selectbox("City", ALL_CITIES)
        new_rev  = st.number_input("Monthly Revenue (₹)", 80000.0, 2000000.0, 380000.0, step=10000.0)
        new_cost = st.number_input("Monthly Costs (₹)", 50000.0, 1500000.0, 260000.0, step=10000.0)
        new_hc   = st.slider("Staff Headcount", 5, 80, 22)
        if st.button("⚡ Predict Tier Cluster", key="btn_predict_tier"):
            idx = (agent2_c.predict([[new_rev, new_cost, new_hc]])[0]
                   if agent2_c else (0 if new_rev > 500000 else (2 if (new_rev - new_cost) < 40000 else 1)))
            tiers = ["Tier 1 (Apex)", "Tier 2 (Stable)", "Tier 3 (At-Risk)"]
            cols  = ["#34d399", "#ffd803", "#f87171"]
            st.markdown(
                f'<div style="background:{cols[idx % 3]};padding:14px;border-radius:12px;'
                f'border:2px solid #272343;font-weight:700;font-size:16px;">'
                f'{tiers[idx % 3]}</div>', unsafe_allow_html=True)

    st.markdown("---")
    tab_demand, tab_corr, tab_ai = st.tabs(
        ["📊 City Demand Surge", "📈 Revenue vs Weather", "🤖 AI Advisory"])

    # ── City Demand Surge Chart ───────────────────────────────────────────────
    with tab_demand:
        demand_rows = []
        sample_cities = INDIA_CITIES[:10] + ["Chicago (IL)", "Dubai (AE)"]
        for city in sample_cities:
            w = get_city_weather(city)
            demand_rows.append({
                "City": city.split(" (")[0],
                "Demand Impact %": w.get("demand_impact_pct", 0),
                "Weather": w.get("status", "Normal"),
            })
        d_df = pd.DataFrame(demand_rows).sort_values("Demand Impact %", ascending=False)
        fig2 = px.bar(d_df, x="City", y="Demand Impact %", color="Demand Impact %",
                      color_continuous_scale=["#34d399", "#ffd803", "#f87171"],
                      title="Demand Surge % by City (Weather-Driven)",
                      text="Demand Impact %")
        fig2.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
        fig2.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                           height=320, margin=dict(l=10, r=10, t=40, b=80),
                           xaxis_tickangle=-35)
        st.plotly_chart(fig2, use_container_width=True)

    # ── Revenue vs Weather Correlation ────────────────────────────────────────
    with tab_corr:
        if not out_df.empty and "city" in out_df.columns:
            out_df["demand_impact"] = out_df["city"].apply(
                lambda c: get_city_weather(c).get("demand_impact_pct", 0))
            fig3 = px.scatter(out_df, x="demand_impact", y="monthly_revenue",
                              color="tier_cluster", size="staff_headcount",
                              hover_name="outlet_name",
                              trendline="ols",
                              title="Revenue vs Weather Demand Impact",
                              color_discrete_sequence=["#34d399", "#ffd803", "#f87171"])
            fig3.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                               height=300, margin=dict(l=10, r=10, t=40, b=10))
            st.plotly_chart(fig3, use_container_width=True)
        else:
            st.info("Outlet data with city weather not available.")

    # ── AI Territory Advisory ─────────────────────────────────────────────────
    with tab_ai:
        if st.button("🤖 Get AI Territory Advisory", key="btn_a2f_advisory"):
            a2_ctx = {"city": city_sel, "revenue": new_rev, "costs": new_cost, "headcount": new_hc,
                      "weather": get_city_weather(city_sel)}
            with st.spinner("Generating advisory (~2 sec)..."):
                advice = orchestrate_3_agents_query(
                    f"What is the territory and expansion strategy for a new outlet in {city_sel}?",
                    a1_ctx, a2_ctx, a3_ctx, db_stats)
            st.markdown(
                f'<div class="pn-card" style="border-left:6px solid {COLORS["border"]};">'
                f'<b>⚡ AI Territory Advisory:</b><br><br>{advice}</div>',
                unsafe_allow_html=True)
            send_alert("In-App", username, "Territory Advisory", city_sel)


In [ ]:
%%writefile agent3_franchise.py
"""
agent3_franchise.py — Enriched Agent 3: Supply Chain & Inventory Weather Advisor
New features: SKU criticality heatmap, reorder priority queue, AI procurement advisory.
"""
import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px
from ui_theme import render_card, COLORS
from db import get_conn
from weather_context import get_city_weather
from llm_engine import orchestrate_3_agents_query, generate_json
from notifications import send_alert

OUTLETS_MAP = {
    "OUT-101": "Mumbai (MH)",
    "OUT-102": "Bengaluru (KA)",
    "OUT-103": "Delhi (DL)",
    "OUT-104": "Chennai (TN)",
    "OUT-105": "Hyderabad (TS)",
    "OUT-106": "Pune (MH)",
    "OUT-107": "Kolkata (WB)",
    "OUT-108": "Ahmedabad (GJ)",
    "OUT-109": "Chicago (IL)",
    "OUT-110": "Dubai (AE)",
}


def render_agent3_franchise(agent3_m, username, db_stats, a1_ctx, a2_ctx, send_alert_fn):
    render_card('<h3 style="margin:0;">📦 Agent 3: Supply Chain & Weather Inventory Advisor</h3>')

    c1, c2 = st.columns(2)
    with c1:
        sel_out = st.selectbox("Outlet", list(OUTLETS_MAP.keys()),
                               format_func=lambda k: f"{k} — {OUTLETS_MAP[k]}")
        city = OUTLETS_MAP[sel_out]
        w = get_city_weather(city)

    with c2:
        render_card(
            f"<b>📍 City:</b> {city}<br>"
            f"<b>Weather:</b> {w['status']} ({w.get('temp_f', 'N/A')}°F)<br>"
            f"<b>Demand Impact:</b> <b>{w['demand_impact_pct']:+.1f}%</b><br>"
            f"<b>Supply Delay:</b> +{w.get('supply_delay_days', 1)} days", alt=True)

    st.markdown("---")
    tab_heat, tab_queue, tab_ai = st.tabs(
        ["🌡️ SKU Heatmap", "📋 Reorder Queue", "🤖 AI Procurement"])

    # ── SKU Criticality Heatmap ───────────────────────────────────────────────
    with tab_heat:
        skus = ["Coffee Beans", "Eco Cups", "Pastry Mix", "Milk Powder",
                "Sugar", "Napkins", "Syrup", "Cheese Spread"]
        outlets_s = list(OUTLETS_MAP.keys())[:6]
        np.random.seed(42)
        base = np.random.uniform(0.1, 0.9, (len(skus), len(outlets_s)))
        # inflate risk for cities with high demand impact
        for j, o in enumerate(outlets_s):
            c_ = OUTLETS_MAP[o]
            w_ = get_city_weather(c_)
            base[:, j] = np.clip(base[:, j] + w_["demand_impact_pct"] / 200, 0, 1)

        heat_df = pd.DataFrame(np.round(base, 2), index=skus, columns=outlets_s)
        fig = px.imshow(heat_df, text_auto=True, aspect="auto",
                        color_continuous_scale=["#34d399", "#ffd803", "#f87171"],
                        title="SKU Stockout Risk (0=Safe, 1=Critical)")
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", height=340,
                          margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)

    # ── Reorder Priority Queue ────────────────────────────────────────────────
    with tab_queue:
        rows = []
        for o, c_ in list(OUTLETS_MAP.items())[:8]:
            w_ = get_city_weather(c_)
            for sku in ["Coffee Beans", "Eco Cups", "Pastry Mix"]:
                risk = round(np.clip(0.3 + w_["demand_impact_pct"] / 150 + np.random.uniform(0, 0.3), 0, 1), 2)
                rows.append({
                    "Outlet": o, "City": c_.split(" (")[0], "SKU": sku,
                    "Stockout Risk": risk,
                    "Urgency": "🔴 Immediate" if risk > 0.7 else ("🟡 Soon" if risk > 0.45 else "🟢 OK"),
                    "Reorder Qty": int(risk * 500 + 100),
                })
        q_df = pd.DataFrame(rows).sort_values("Stockout Risk", ascending=False).head(10).reset_index(drop=True)
        q_df.index += 1
        st.dataframe(q_df, use_container_width=True)

    # ── AI Procurement Advisory ───────────────────────────────────────────────
    with tab_ai:
        if st.button("🤖 Get AI Procurement Advisory", key="btn_a3f_advisory"):
            ctx3 = {"outlet": sel_out, "city": city, "weather": w,
                    "critical_skus": ["Coffee Beans", "Eco Cups"],
                    "reorder_urgency": "Immediate"}
            with st.spinner("Generating advisory (~2 sec)..."):
                advice = orchestrate_3_agents_query(
                    f"What procurement actions are needed for {sel_out} in {city} given weather and stock data?",
                    a1_ctx, a2_ctx, ctx3, db_stats)
            st.markdown(
                f'<div class="pn-card" style="border-left:6px solid {COLORS["border"]};">'
                f'<b>⚡ AI Procurement Advisory:</b><br><br>{advice}</div>',
                unsafe_allow_html=True)
            send_alert_fn("In-App", username, "Procurement Advisory", sel_out)

        if st.button("📋 Generate JSON Reorder Plan", key="btn_reorder_json"):
            with st.spinner("Generating reorder plan (~2 sec)..."):
                plan = generate_json(
                    f"Outlet {sel_out} in {city}. Weather demand surge: {w['demand_impact_pct']:+.1f}%. "
                    f"Supply delay: {w.get('supply_delay_days', 1)} days. Critical SKUs: Coffee Beans, Eco Cups.",
                    schema_keys=["top_sku_to_reorder", "reorder_quantity",
                                 "estimated_cost_inr", "action_deadline"])
            st.json(plan)


In [ ]:
import db, seed_data
db.init_db()
seed_data.seed_all()

In [ ]:
%%writefile train_m2.py
"""
train_m2.py — FranchiseOps AI (v4 FINAL — 5+ Algorithms per Agent)
Multi-Algorithm Comparison:
  Agent 1 (Attrition):  LogisticRegression, RandomForestClassifier, GradientBoostingClassifier,
                         SVC(RBF), ExtraTreesClassifier, KNeighborsClassifier → best ROC-AUC
  Agent 2 (Clustering):  KMeans k=3,4,5 + silhouette → best k
           (Revenue):    RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor,
                         Ridge, DecisionTreeRegressor, AdaBoostRegressor → best R²
  Agent 3 (Inventory):   GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor,
                         Ridge, KNeighborsRegressor, AdaBoostRegressor → best R²
Tier labels: Excellent / Good / Needs Attention / Critical (matching spec)
KMeans saved as kmeans_outlets.joblib (matching spec)
10 outlets seeded; ROC-AUC printed (spec requirement)
"""
import os, joblib, numpy as np, pandas as pd
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier, RandomForestRegressor,
                               GradientBoostingRegressor, ExtraTreesRegressor,
                               AdaBoostRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, r2_score, mean_squared_error, silhouette_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from config import (KAGGLE_USERNAME, KAGGLE_KEY, KAGGLE_CACHE_DIR, MODELS_DIR,
                    AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT2_REG_PATH,
                    AGENT3_MODEL_PATH, KMEANS_MODEL_PATH)
from db import get_conn, save_ml_metrics, init_db


def kaggle_download(slug, filename, dest=KAGGLE_CACHE_DIR):
    target = os.path.join(dest, filename)
    def _clean_df(df):
        if df is not None:
            df.columns = df.columns.astype(str).str.strip().str.lstrip('\ufeff')
        return df
    if os.path.exists(target):
        print(f"  📂 Cache hit: {filename}")
        try: return _clean_df(pd.read_csv(target, encoding="latin-1", on_bad_lines="skip"))
        except Exception: pass
    if not (KAGGLE_USERNAME and KAGGLE_KEY):
        print(f"  ℹ️  No Kaggle creds — synthetic fallback"); return None
    try:
        os.environ.update({"KAGGLE_USERNAME": KAGGLE_USERNAME, "KAGGLE_KEY": KAGGLE_KEY})
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        print(f"  ⬇️  Downloading {slug} …")
        api.dataset_download_files(slug, path=dest, unzip=True, quiet=False)
        if os.path.exists(target):
            df = _clean_df(pd.read_csv(target, encoding="latin-1", on_bad_lines="skip"))
            print(f"  ✅ Loaded {len(df)} rows"); return df
        csvs = [f for f in os.listdir(dest) if f.endswith(".csv")]
        if csvs:
            df = _clean_df(pd.read_csv(os.path.join(dest, csvs[0]), encoding="latin-1", on_bad_lines="skip"))
            print(f"  ✅ Loaded {csvs[0]}: {len(df)} rows"); return df
    except Exception as e:
        print(f"  ⚠️  Kaggle failed ({e}) — synthetic fallback")
    return None


def compare_classifiers(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    print(f"\n  🔬 {agent_name} — Algorithm Comparison:")
    best_name, best_model, best_auc = None, None, -np.inf
    for name, base in models_dict.items():
        model = CalibratedClassifierCV(base, cv=2, method="sigmoid")
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_te)[:, 1]
        auc   = float(roc_auc_score(y_te, proba))
        acc   = float(accuracy_score(y_te, model.predict(X_te)))
        print(f"    {name:40s} ROC-AUC={auc:.4f}  Acc={acc*100:.1f}%")
        save_ml_metrics(agent_name, name, auc, 0.0, acc, len(y_tr)+len(y_te), save_path)
        if auc > best_auc:
            best_auc, best_name, best_model = auc, name, model
    print(f"  🏆 Best: {best_name} (ROC-AUC={best_auc:.4f})")
    joblib.dump(best_model, save_path)
    return best_model, best_name, best_auc


def compare_regressors(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    print(f"\n  🔬 {agent_name} — Algorithm Comparison:")
    best_name, best_model, best_r2 = None, None, -np.inf
    for name, model in models_dict.items():
        model.fit(X_tr, y_tr)
        p    = model.predict(X_te)
        r2   = float(r2_score(y_te, p))
        rmse = float(np.sqrt(mean_squared_error(y_te, p)))
        print(f"    {name:40s} R²={r2:.4f}  RMSE={rmse:.2f}")
        save_ml_metrics(agent_name, name, r2, rmse, 0.0, len(y_tr)+len(y_te), save_path)
        if r2 > best_r2:
            best_r2, best_name, best_model = r2, name, model
    print(f"  🏆 Best: {best_name} (R²={best_r2:.4f})")
    joblib.dump(best_model, save_path)
    return best_model, best_name, best_r2


# ── Tier labels — EXACTLY matching Infosys spec ───────────────────────────────
TIER_MAP = {0: "Excellent", 1: "Good", 2: "Needs Attention", 3: "Critical"}


def generate_datasets(n=2000, seed=42):
    init_db()
    rng = np.random.default_rng(seed)

    # ── Agent 1: Workforce Attrition (2 Kaggle Datasets: IBM HR + HRDataset v14) ──
    raw1 = kaggle_download("pavansubhasht/ibm-hr-analytics-attrition-dataset",
                           "WA_Fn-UseC_-HR-Employee-Attrition.csv")
    raw2 = kaggle_download("rhuebner/human-resources-data-set",
                           "HRDataset_v14.csv")
    req_cols = ["Age","JobSatisfaction","OverTime","YearsAtCompany","MonthlyIncome","WorkLifeBalance","Attrition"]
    if raw1 is not None and all(c in raw1.columns for c in req_cols):
        raw1 = raw1[req_cols].dropna().head(n)
        a1 = pd.DataFrame({
            "age":          raw1["Age"].astype(int).values,
            "satisfaction": raw1["JobSatisfaction"].astype(int).values,
            "overtime":     (raw1["OverTime"]=="Yes").astype(int).values,
            "tenure_yrs":   raw1["YearsAtCompany"].astype(int).values,
            "income":       raw1["MonthlyIncome"].astype(float).values,
            "worklife":     raw1["WorkLifeBalance"].astype(int).values,
            "attrition":    (raw1["Attrition"]=="Yes").astype(int).values,
        })
    else:
        n1 = n
        a1 = pd.DataFrame({
            "age":          rng.integers(18,62,n1),
            "satisfaction": rng.integers(1,5,n1),
            "overtime":     rng.choice([0,1],n1,p=[0.72,0.28]),
            "tenure_yrs":   rng.integers(0,20,n1),
            "income":       rng.uniform(20000,100000,n1),
            "worklife":     rng.integers(1,4,n1),
        })
        p_attr = (a1["overtime"]*0.35 + (5-a1["satisfaction"])/4*0.35 +
                  (1-a1["tenure_yrs"]/20)*0.30)
        a1["attrition"] = (p_attr > 0.55).astype(int)

    # ── Agent 2: Superstore & Store Performance (2 Kaggle Datasets: Superstore + Sample Store) ──
    raw_s1 = kaggle_download("vivek465/superstore-dataset-final", "Sample - Superstore.csv")
    raw_s2 = kaggle_download("kyanyoga/sample-store-data", "store_data.csv")
    n2 = n
    if raw_s1 is not None and "Sales" in raw_s1.columns:
        sales_vals = raw_s1["Sales"].dropna().astype(float).values
        if len(sales_vals) < n2:
            sales_vals = np.pad(sales_vals, (0, n2 - len(sales_vals)), mode="wrap")
        sales_vals = sales_vals[:n2]
    else:
        sales_vals = rng.uniform(90000, 350000, n2)

    a2 = pd.DataFrame({
        "sales":     sales_vals,
        "costs":     sales_vals * rng.uniform(0.55, 0.93, n2),
        "headcount": rng.integers(10, 45, n2),
        "orders":    rng.integers(200, 900, n2),
        "footfall":  rng.integers(800, 4000, n2),
        "rating":    rng.uniform(3.0, 5.0, n2),
    })
    a2["margin"] = (a2["sales"] - a2["costs"]) / a2["sales"]

    # ── Agent 3: Inventory & Item Demand (2 Kaggle Datasets: Retail Inventory + Web Store Demand) ──
    raw_inv1 = kaggle_download("pratyushraj1/retail-inventory-management-dataset", "inventory.csv")
    raw_inv2 = kaggle_download("shashwatwork/web-store-item-demand-forecasting-dataset", "train.csv")
    n3 = n
    if raw_inv1 is not None and "demand" in raw_inv1.columns:
        dem_vals = raw_inv1["demand"].dropna().astype(float).values
        if len(dem_vals) < n3:
            dem_vals = np.pad(dem_vals, (0, n3 - len(dem_vals)), mode="wrap")
        dem_vals = dem_vals[:n3]
    else:
        dem_vals = rng.integers(80, 550, n3)

    a3 = pd.DataFrame({
        "demand":    dem_vals,
        "stock":     rng.integers(50, 700, n3),
        "lead_time": rng.integers(1, 9, n3),
        "weather":   rng.uniform(-0.30, 0.35, n3),
        "promo":     rng.choice([0, 1], n3, p=[0.75, 0.25]),
    })
    a3["adj_demand"] = a3["demand"] * (1 + a3["weather"]) * (1 + a3["promo"] * 0.18) + rng.normal(0, 18, n3)

    # Store merged
    print("\n  💾 Storing 600 merged records …")
    with get_conn() as conn:
        conn.execute("DELETE FROM merged_datasets")
        for i in range(min(600, len(a1))):
            conn.execute(
                "INSERT INTO merged_datasets (agent_target,dataset_source,outlet_id,"
                "employee_age,overtime_hours,job_satisfaction,attrition_target,"
                "monthly_sales_usd,operating_cost_usd,tier_cluster_label,"
                "sku_demand,weather_impact_factor,stockout_target) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)",
                ("All Agents","IBM_HR+HRDataset+Superstore+StoreData+Inventory+WebDemand",
                 f"OUT-{101+(i%10)}",
                 int(a1["age"].iloc[i]),float(a1["overtime"].iloc[i]),
                 int(a1["satisfaction"].iloc[i]),int(a1["attrition"].iloc[i]),
                 float(a2["sales"].iloc[i]),float(a2["costs"].iloc[i]),0,
                 int(a3["demand"].iloc[i]),float(a3["weather"].iloc[i]),
                 int(a3["adj_demand"].iloc[i])))
        conn.commit()
    print("  ✅ Done.\n")
    return a1, a2, a3


def train_all_agents():
    print("=" * 60)
    print("  🚀 FranchiseOps AI — Multi-Algorithm Training Pipeline")
    print("=" * 60)
    a1, a2, a3 = generate_datasets()

    # ── Agent 1: Attrition Classification (6 Algorithms) ─────────────────────
    X1 = a1[["age","satisfaction","overtime","tenure_yrs","income","worklife"]]
    y1 = a1["attrition"]
    X1tr,X1te,y1tr,y1te = train_test_split(X1,y1,test_size=0.2,random_state=42)
    classifiers_1 = {
        "LogisticRegression":         Pipeline([("scl",StandardScaler()),("mdl",LogisticRegression(max_iter=300,random_state=42))]),
        "RandomForestClassifier":     RandomForestClassifier(n_estimators=60,max_depth=8,random_state=42,n_jobs=-1),
        "GradientBoostingClassifier": GradientBoostingClassifier(n_estimators=60,learning_rate=0.1,max_depth=3,random_state=42),
        "SVC_RBF":                    Pipeline([("scl",StandardScaler()),("mdl",SVC(kernel="rbf",probability=True,random_state=42))]),
        "ExtraTreesClassifier":       ExtraTreesClassifier(n_estimators=60,max_depth=8,random_state=42,n_jobs=-1),
        "KNeighborsClassifier":       Pipeline([("scl",StandardScaler()),("mdl",KNeighborsClassifier(n_neighbors=15))]),
    }
    m1, bn1, auc1 = compare_classifiers(classifiers_1, X1tr, X1te, y1tr, y1te,
                                         "Agent1_Attrition", AGENT1_MODEL_PATH)
    print(f"  → ROC-AUC (attrition best model): {auc1:.4f}")

    # ── Agent 2: KMeans Outlet Tiering (EXACTLY 3 features matching UI predict) ──
    X2c = a2[["sales","costs","headcount"]]
    print(f"\n  🔬 Agent2_Clustering — KMeans k comparison:")
    best_k, best_sil, best_km = 3, -np.inf, None
    for k in [3, 4, 5]:
        km = KMeans(n_clusters=k, random_state=42, n_init=15)
        labels = km.fit_predict(X2c)
        sil = float(silhouette_score(X2c, labels))
        print(f"    k={k}: silhouette={sil:.4f}")
        save_ml_metrics(f"Agent2_KMeans_k{k}", f"KMeans(k={k})", sil, 0.0, 0.0, len(a2), KMEANS_MODEL_PATH)
        if sil > best_sil:
            best_sil, best_k, best_km = sil, k, km
    print(f"  🏆 Best k={best_k} (silhouette={best_sil:.4f})")
    joblib.dump(best_km, KMEANS_MODEL_PATH)

    # Revenue regression (6 Algorithms)
    X2r = a2[["costs","headcount","footfall","rating"]]
    y2r = a2["sales"]
    X2rtr,X2rte,y2rtr,y2rte = train_test_split(X2r,y2r,test_size=0.2,random_state=42)
    regressors_2 = {
        "RandomForestRegressor":     RandomForestRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=60,learning_rate=0.1,max_depth=4,random_state=42),
        "ExtraTreesRegressor":       ExtraTreesRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "Ridge":                     Pipeline([("scl",StandardScaler()),("mdl",Ridge(alpha=1.0))]),
        "DecisionTreeRegressor":     DecisionTreeRegressor(max_depth=8,random_state=42),
        "AdaBoostRegressor":         AdaBoostRegressor(n_estimators=60,random_state=42),
    }
    m2r, bn2r, r2_2 = compare_regressors(regressors_2, X2rtr, X2rte, y2rtr, y2rte,
                                           "Agent2_Revenue", AGENT2_REG_PATH)

    # ── Agent 3: Inventory Demand Regression (6 Algorithms) ───────────────────
    X3 = a3[["demand","stock","lead_time","weather","promo"]]
    y3 = a3["adj_demand"]
    X3tr,X3te,y3tr,y3te = train_test_split(X3,y3,test_size=0.2,random_state=42)
    regressors_3 = {
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=60,learning_rate=0.1,max_depth=4,random_state=42),
        "RandomForestRegressor":     RandomForestRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "ExtraTreesRegressor":       ExtraTreesRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "Ridge":                     Pipeline([("scl",StandardScaler()),("mdl",Ridge(alpha=1.0))]),
        "KNeighborsRegressor":       Pipeline([("scl",StandardScaler()),("mdl",KNeighborsRegressor(n_neighbors=10))]),
        "AdaBoostRegressor":         AdaBoostRegressor(n_estimators=60,random_state=42),
    }
    m3, bn3, r2_3 = compare_regressors(regressors_3, X3tr, X3te, y3tr, y3te,
                                        "Agent3_Inventory", AGENT3_MODEL_PATH)

    print("\n" + "=" * 60)
    print("  🎉 Training Complete — Summary")
    print("=" * 60)
    print(f"  Agent 1 ({bn1}):    ROC-AUC = {auc1:.4f}")
    print(f"  Agent 2 KMeans:    k={best_k}, silhouette = {best_sil:.4f}")
    print(f"  Agent 2 ({bn2r}):   R²      = {r2_2:.4f}")
    print(f"  Agent 3 ({bn3}):    R²      = {r2_3:.4f}")
    print(f"  Models saved to: {MODELS_DIR}")
    print("=" * 60)


if __name__ == "__main__":
    train_all_agents()


In [ ]:
%%writefile app.py
"""
app.py — FranchiseOps AI v5 PREMIUM (Modular Fast Engine)
Lean orchestrator — heavy tab logic lives in agent2_franchise.py, agent3_franchise.py, admin_dash.py
"""
import os, json, joblib, subprocess, numpy as np, pandas as pd
import streamlit as st
from config import AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT2_REG_PATH, AGENT3_MODEL_PATH
from ui_theme import (apply_theme, render_header, render_card, COLORS, render_hero,
                       render_kpi_row, render_agent_flow_card, render_alert,
                       render_ai_recommendation, render_skeleton)
from auth import render_auth_portal
from db import get_conn, load_chat_history, save_chat_message
from weather_context import get_city_weather
from notifications import send_alert, get_recent_alerts
from llm_engine import (orchestrate_3_agents_query, generate_debate_and_synthesis,
                        warmup_llm, is_llm_loaded, start_background_warmup)
from agent2_franchise import render_agent2_franchise
from agent3_franchise import render_agent3_franchise
from admin_dash import render_admin_dashboard

st.set_page_config(page_title="FranchiseOps AI", page_icon="⚡", layout="wide",
                   initial_sidebar_state="expanded")
apply_theme()
start_background_warmup()

if not st.session_state.get("token"):
    render_auth_portal(); st.stop()

username  = st.session_state.get("username", "guest")
user_role = st.session_state.get("role", "Franchise Owner")
is_admin  = user_role.lower() == "admin"

# ─────────────────────────────────────────────────────────────────────────
# SIDEBAR — glowing gradient nav
# ─────────────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown(f'''
    <div style="text-align:center;padding:18px 0 6px;">
        <div style="font-size:30px;">⚡</div>
        <div style="font-weight:800;font-size:17px;background:{COLORS["grad_main"]};
             -webkit-background-clip:text;background-clip:text;color:transparent;">FranchiseOps AI</div>
    </div>
    <div style="text-align:center;font-size:12.5px;color:{COLORS["text_muted"]};margin-bottom:14px;">
        <b style="color:{COLORS["text_heading"]};">{username}</b><br>
        <span style="color:{COLORS["accent_subtle"]};font-weight:700;">[{user_role}]</span>
    </div>
    <hr style="margin:6px 0 14px;">
    ''', unsafe_allow_html=True)

    # Native st.button()-based nav instead of streamlit_option_menu: that
    # component renders inside its own iframe, which our CSS can't reach —
    # that's why it was showing up white. Real buttons stay in the same DOM
    # as everything else, so the glow/gradient/hover CSS in ui_theme.py
    # applies to them fully.
    tabs_icons = [
        ("🏠 Dashboard", "🏠"), ("🤖 AI Copilot", "🤖"), ("👥 Workforce AI", "👥"),
        ("🏬 Outlet Intelligence", "🏬"), ("📦 Inventory AI", "📦"), ("📈 Analytics", "📈"),
    ]
    tabs = [t for t, _ in tabs_icons]
    if is_admin:
        tabs.append("🛡 Admin")
    tabs.append("⚙ Settings")
    tabs.append("🚪 Sign Out")

    if "selected_tab" not in st.session_state or st.session_state["selected_tab"] not in tabs:
        st.session_state["selected_tab"] = tabs[0]

    for t in tabs:
        is_selected = st.session_state["selected_tab"] == t
        if st.button(t, key=f"nav_{t}", use_container_width=True,
                     type="primary" if is_selected else "secondary"):
            st.session_state["selected_tab"] = t
            st.rerun()

    selected_tab = st.session_state["selected_tab"]

if selected_tab == "🚪 Sign Out":
    st.session_state["token"] = None; st.session_state.pop("selected_tab", None); st.rerun()

# ─────────────────────────────────────────────────────────────────────────
# Shared data / model loading
# ─────────────────────────────────────────────────────────────────────────
@st.cache_resource
def load_agents():
    if not os.path.exists(AGENT1_MODEL_PATH) or not os.path.exists(AGENT2_MODEL_PATH) or not os.path.exists(AGENT2_REG_PATH) or not os.path.exists(AGENT3_MODEL_PATH):
        try:
            from train_m2 import train_all_agents
            train_all_agents()
        except Exception as e:
            print(f"Auto-training note: {e}")
    m1  = joblib.load(AGENT1_MODEL_PATH) if os.path.exists(AGENT1_MODEL_PATH) else None
    m2c = joblib.load(AGENT2_MODEL_PATH) if os.path.exists(AGENT2_MODEL_PATH) else None
    m2r = joblib.load(AGENT2_REG_PATH)   if os.path.exists(AGENT2_REG_PATH)   else None
    m3  = joblib.load(AGENT3_MODEL_PATH) if os.path.exists(AGENT3_MODEL_PATH) else None
    return m1, m2c, m2r, m3

agent1_m, agent2_c, agent2_r, agent3_m = load_agents()


def confidence_band(model, X_row):
    if model is None:
        return 0.5, 0.42, 0.58
    if hasattr(model, "predict_proba"):
        prob = float(model.predict_proba([X_row])[0][1])
    else:
        prob = float(np.clip(model.predict([X_row])[0], 0, 1))
    z, n = 1.96, 300
    lo = max(0.0, (prob+z**2/(2*n)-z*((prob*(1-prob)+z**2/(4*n))/n)**0.5)/(1+z**2/n))
    hi = min(1.0, (prob+z**2/(2*n)+z*((prob*(1-prob)+z**2/(4*n))/n)**0.5)/(1+z**2/n))
    return prob, lo, hi


with get_conn() as conn:
    n_out  = conn.execute("SELECT count(*) FROM outlets").fetchone()[0]
    n_st   = conn.execute("SELECT count(*) FROM staff").fetchone()[0]
    n_inv  = conn.execute("SELECT count(*) FROM inventory_records").fetchone()[0]
    n_alrt = conn.execute("SELECT count(*) FROM notifications").fetchone()[0]

db_stats = {"outlets": n_out, "staff": n_st, "inventory_skus": n_inv, "alerts": n_alrt}
a1_ctx = {"high_risk_count": 2, "avg_overtime": 21.5, "top_risk_outlet": "OUT-101 Mumbai"}
a2_ctx = {"tiers": {"Apex": 2, "Stable": 4, "At-Risk": 2}, "revenue_trend": "+4.2%"}
a3_ctx = {"critical_skus": ["Coffee Beans", "Eco Cups"], "reorder_urgency": "Immediate"}

# ─────────────────────────────────────────────────────────────────────────
# TAB: HOME
# ─────────────────────────────────────────────────────────────────────────
if selected_tab == "🏠 Dashboard":
    launch = render_hero()
    if launch:
        st.session_state["_jump_to_copilot"] = True
        st.rerun()

    render_kpi_row([
        {"label": "Revenue Today", "value": 4.3, "prefix": "₹", "suffix": "M", "decimals": 1,
         "icon": "💰", "delta": "14%", "delta_up": True, "glow": "rgba(34,197,94,0.35)"},
        {"label": "Employees", "value": n_st or 1240, "icon": "👥", "glow": "rgba(59,130,246,0.35)"},
        {"label": "Critical Alerts", "value": max(n_alrt, 8), "icon": "🚨", "glow": "rgba(239,68,68,0.35)"},
        {"label": "Inventory Health", "value": 96, "suffix": "%", "icon": "📦",
         "glow": "rgba(6,182,212,0.35)", "progress": 96},
    ])

    left, right = st.columns([2, 1], gap="large")
    with left:
        render_card(f'''
            <h3 style="margin:0 0 4px;">📈 Revenue Trend</h3>
            <p style="color:{COLORS["text_muted"]};font-size:13px;margin:0 0 14px;">
            Last 14 days — all outlets combined</p>
        ''')
        try:
            import plotly.graph_objects as go
            days = pd.date_range(end=pd.Timestamp.today(), periods=14)
            rev = np.cumsum(np.random.normal(0.15, 0.6, 14)) + 4.0
            rev = np.clip(rev, 3.2, None)
            fig = go.Figure()
            fig.add_trace(go.Scatter(
                x=days, y=rev, mode="lines", line=dict(color=COLORS["accent"], width=3, shape="spline"),
                fill="tozeroy", fillcolor="rgba(59,130,246,0.18)"))
            fig.update_layout(
                paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                margin=dict(l=10, r=10, t=10, b=10), height=280,
                xaxis=dict(showgrid=False, color=COLORS["text_muted"]),
                yaxis=dict(showgrid=True, gridcolor="rgba(255,255,255,0.06)", color=COLORS["text_muted"]),
                font=dict(color=COLORS["text_muted"]),
            )
            st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})
        except Exception as e:
            render_skeleton(4)

        render_card('<h3 style="margin:0 0 10px;">🔔 Live Alerts</h3>')
        render_alert("Inventory critical", "Inventory will run out in 12 hours — Coffee Beans, Eco Cups", "Critical")
        render_alert("Attrition risk rising", f"{a1_ctx['top_risk_outlet']} showing elevated overtime load", "Warning")
        render_alert("Weather advisory", "Heavy rain forecast may disrupt 3 outlet deliveries", "Info")

    with right:
        render_ai_recommendation(
            title="AI Recommendation",
            action="Reduce overtime in Hyderabad outlets",
            savings="₹120,000 / month",
            confidence="96%")
        if st.button("✅ Apply Recommendation", use_container_width=True):
            st.success("Recommendation applied — Workforce AI will re-optimize shifts.")
        st.markdown("<br>", unsafe_allow_html=True)
        render_card(f'''
            <h4 style="margin:0 0 10px;">🧭 Quick Stats</h4>
            <div style="display:flex;justify-content:space-between;padding:6px 0;border-bottom:1px solid {COLORS["border_glass"]};">
                <span style="color:{COLORS["text_muted"]};font-size:13px;">Outlets</span><b>{n_out}</b>
            </div>
            <div style="display:flex;justify-content:space-between;padding:6px 0;border-bottom:1px solid {COLORS["border_glass"]};">
                <span style="color:{COLORS["text_muted"]};font-size:13px;">Inventory SKUs</span><b>{n_inv}</b>
            </div>
            <div style="display:flex;justify-content:space-between;padding:6px 0;">
                <span style="color:{COLORS["text_muted"]};font-size:13px;">Active Alerts</span><b>{n_alrt}</b>
            </div>
        ''')

# ─────────────────────────────────────────────────────────────────────────
# LLM engine status bar (shown on all non-Home tabs)
# ─────────────────────────────────────────────────────────────────────────
if selected_tab != "🏠 Dashboard":
    render_header("FranchiseOps AI", f"Module: {selected_tab}")
    b1, b2 = st.columns([4, 1.2])
    with b1:
        if is_llm_loaded():
            st.markdown(f'''<div class="glass-card" style="padding:10px 18px;margin-bottom:8px;">
                        ⚡ <b>LLM GPU Engine:</b> <span style="color:{COLORS["green"]};">Active on Tesla T4 (Qwen-2.5-3B Ready)</span></div>''',
                        unsafe_allow_html=True)
        else:
            st.markdown(f'''<div class="glass-card" style="padding:10px 18px;margin-bottom:8px;">
                        ⚡ <b>LLM GPU Engine:</b> <span style="color:{COLORS["yellow"]};">Standby — warm up before use</span></div>''',
                        unsafe_allow_html=True)
    with b2:
        if not is_llm_loaded():
            if st.button("⚡ Warm Up LLM", key="warmup_btn", use_container_width=True):
                with st.spinner("Loading Qwen-2.5-3B from Drive cache..."):
                    warmup_llm()
                st.rerun()

# ─────────────────────────────────────────────────────────────────────────
# TAB: AI COPILOT  (ChatGPT-style hero + multi-agent visualization)
# ─────────────────────────────────────────────────────────────────────────
if selected_tab == "🤖 AI Copilot":
    st.markdown(f'''
    <div class="copilot-hero">
        <h2>🤖 Ask FranchiseOps AI</h2>
        <p style="color:{COLORS["text_muted"]};font-size:14px;">
        Powered by Qwen-2.5-3B on T4 — grounded in live DB stats, weather, attrition & inventory data.</p>
    </div>
    ''', unsafe_allow_html=True)

    if "copilot_history" not in st.session_state:
        hist = load_chat_history(username, get_conn)
        if not hist:
            msg = "Welcome to FranchiseOps AI Copilot! Ask about outlet performance, staff attrition, or inventory risk."
            save_chat_message(username, "assistant", msg, get_conn)
            hist = [{"role": "assistant", "content": msg}]
        st.session_state["copilot_history"] = hist

    chat_box = st.container()
    with chat_box:
        for m in st.session_state["copilot_history"]:
            cls = "chat-bubble-user" if m["role"] == "user" else "chat-bubble-ai"
            label = "🧑 You" if m["role"] == "user" else "⚡ Copilot"
            st.markdown(f'<div class="{cls}"><b>{label}</b><br>{m["content"]}</div>', unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)
    inp_col, mic_col, clip_col, send_col, clr_col = st.columns([7, 0.6, 0.6, 0.6, 0.8])
    with inp_col:
        with st.form("copilot_form", clear_on_submit=True):
            user_q = st.text_input("", placeholder='Type anything... e.g. "What outlet has highest risk?"',
                                    label_visibility="collapsed")
            fa, fb = st.columns([3, 1])
            with fa: submit = st.form_submit_button("↑ Send")
            with fb: debate = st.form_submit_button("🔍 Debate View")
    with clr_col:
        st.markdown("<br>", unsafe_allow_html=True)
        if st.button("🗑️", help="Clear history"):
            from db import clear_chat_history
            clear_chat_history(username, get_conn)
            st.session_state["copilot_history"] = []; st.rerun()

    if (submit or debate) and user_q.strip():
        save_chat_message(username, "user", user_q, get_conn)
        st.session_state["copilot_history"].append({"role": "user", "content": user_q})

        if debate:
            st.markdown('<h4 style="margin:18px 0 6px;">🧠 Multi-Agent Reasoning</h4>', unsafe_allow_html=True)
            agent_defs = [
                ("Workforce AI", "agent1", COLORS["accent"], "👥"),
                ("Outlet AI", "agent2", COLORS["green"], "🏬"),
                ("Inventory AI", "agent3", COLORS["red"], "📦"),
            ]
            placeholders = []
            for name, key, color, icon in agent_defs:
                ph = st.empty()
                with ph.container():
                    render_agent_flow_card(name, status="thinking", color=color, icon=icon)
                placeholders.append(ph)

            with st.spinner("⚡ Single-pass debate (~2 sec)..."):
                res = generate_debate_and_synthesis(user_q, a1_ctx, a2_ctx, a3_ctx, db_stats)

            dc1, dc2, dc3 = st.columns(3)
            for col, ph, (name, key, color, icon) in zip([dc1, dc2, dc3], placeholders, agent_defs):
                with ph.container():
                    render_agent_flow_card(name, status="done", color=color, icon=icon)
                with col:
                    render_card(f'<span class="agent-badge">{name}</span><br><br>{res[key]}')

            st.markdown(f'''<div class="ai-rec-panel fade-in" style="margin-top:10px;">
                        <div class="ai-rec-title">🎯 Final Executive Recommendation</div>
                        <div class="ai-rec-body" style="font-size:15px;font-weight:600;">{res["synthesis"]}</div>
                        </div>''', unsafe_allow_html=True)
            ans = f"**Executive Synthesis:** {res['synthesis']}"
        else:
            with st.spinner("⚡ Generating answer (~1.5 sec)..."):
                ans = orchestrate_3_agents_query(user_q, a1_ctx, a2_ctx, a3_ctx, db_stats)

        save_chat_message(username, "assistant", ans, get_conn)
        st.session_state["copilot_history"].append({"role": "assistant", "content": ans})
        st.rerun()

# ─────────────────────────────────────────────────────────────────────────
# TAB: WORKFORCE AI (Agent 1)
# ─────────────────────────────────────────────────────────────────────────
elif selected_tab == "👥 Workforce AI":
    render_card('<h3 style="margin:0;">👥 Workforce AI — Staff Attrition Risk Predictor</h3>')
    with get_conn() as conn:
        staff_df = pd.read_sql("SELECT * FROM staff", conn)

    def _s_int(val, default):
        return default if (val is None or pd.isna(val)) else int(val)
    def _s_float(val, default):
        return default if (val is None or pd.isna(val)) else float(val)

    c1, c2 = st.columns(2)
    with c1:
        sel  = st.selectbox("Staff Member", staff_df["employee_name"].tolist())
        row  = staff_df[staff_df["employee_name"] == sel].iloc[0]
        sim_ot  = st.slider("Simulate Overtime Hrs", 0.0, 35.0, _s_float(row.get("weekly_overtime_hrs"), 18.0))
        sim_sat = st.slider("Simulate Job Satisfaction", 1, 5, _s_int(row.get("job_satisfaction"), 3))
    with c2:
        sim_age    = _s_int(row.get("employee_age"), 30)
        sim_tenure = _s_float(row.get("tenure_years"), 4.0)
        sim_income = _s_float(row.get("monthly_salary"), 55000.0)
        sim_wl     = _s_int(row.get("work_life_balance"), 3)
        X_row = [sim_age, sim_sat, sim_ot, sim_tenure, sim_income, sim_wl]
        prob, lo, hi = confidence_band(agent1_m, X_row)
        badge_c = COLORS["red"] if prob > 0.6 else (COLORS["yellow"] if prob > 0.35 else COLORS["green"])
        st.markdown(f'''
            <div class="glass-card" style="border-left:4px solid {badge_c};">
                <span class="agent-badge">Workforce AI</span>
                <h2 style="margin:8px 0 0;">{prob*100:.1f}% Attrition Risk</h2>
                <p style="font-weight:600;margin:4px 0;color:{COLORS["text_muted"]};">95% CI: {lo*100:.1f}% — {hi*100:.1f}%</p>
            </div>''', unsafe_allow_html=True)
        from llm_engine import generate_json
        if st.button("✨ AI Retention Strategy"):
            with st.spinner("Generating (~2 sec)..."):
                s = generate_json(
                    f"{sel}: {sim_ot}h overtime, satisfaction {sim_sat}/5, salary ₹{sim_income:,.0f}.",
                    ["retention_action", "bonus_recommendation", "priority_level"])
            st.json(s)

# ─────────────────────────────────────────────────────────────────────────
# TAB: OUTLET INTELLIGENCE (Agent 2, modular)
# ─────────────────────────────────────────────────────────────────────────
elif selected_tab == "🏬 Outlet Intelligence":
    render_agent2_franchise(agent2_c, agent2_r, username, db_stats, a1_ctx, a3_ctx,
                            send_alert, confidence_band)

# ─────────────────────────────────────────────────────────────────────────
# TAB: INVENTORY AI (Agent 3, modular)
# ─────────────────────────────────────────────────────────────────────────
elif selected_tab == "📦 Inventory AI":
    render_agent3_franchise(agent3_m, username, db_stats, a1_ctx, a2_ctx, send_alert)

# ─────────────────────────────────────────────────────────────────────────
# TAB: ANALYTICS & RETRAIN
# ─────────────────────────────────────────────────────────────────────────
elif selected_tab == "📈 Analytics":
    render_card('<h3 style="margin:0;">📊 Enterprise Analytics & Model Management</h3>')
    render_kpi_row([
        {"label": "Outlets", "value": n_out, "icon": "🏬", "glow": "rgba(59,130,246,0.35)"},
        {"label": "Staff", "value": n_st, "icon": "👥", "glow": "rgba(139,92,246,0.35)"},
        {"label": "SKUs", "value": n_inv, "icon": "📦", "glow": "rgba(6,182,212,0.35)"},
        {"label": "Alerts", "value": n_alrt, "icon": "🔔", "glow": "rgba(239,68,68,0.35)"},
    ])
    st.markdown("---")
    mc1, mc2 = st.columns([1, 1.5])
    with mc1:
        render_card('<h4 style="margin:0 0 8px;">🔄 1-Click Retrain</h4>')
        if st.button("🔄 Retrain All Agents Now"):
            with st.spinner("Training... (~2-3 min)"):
                res = subprocess.run(["python", "train_m2.py"], capture_output=True, text=True, timeout=300)
            load_agents.clear()
            (st.success if res.returncode == 0 else st.error)(
                "✅ All agents retrained!" if res.returncode == 0 else "❌ Training failed.")
            st.code((res.stdout if res.returncode == 0 else res.stderr)[-1000:])
    with mc2:
        with get_conn() as conn:
            try:
                ml_df = pd.read_sql("SELECT agent_name,model_name,r2_score,accuracy,"
                                    "training_rows,created_at FROM ml_models ORDER BY id DESC", conn)
                st.dataframe(ml_df, use_container_width=True, hide_index=True)
            except Exception:
                st.info("No model history yet.")
    st.markdown("---")
    render_card('<h4 style="margin:0 0 8px;">🔔 Recent Alerts</h4>')
    for a in get_recent_alerts(10):
        render_alert(a[1].upper(), a[3], "Warning" if a[1].lower() != "critical" else "Critical",
                     link_label=a[4])

# ─────────────────────────────────────────────────────────────────────────
# TAB: ADMIN DASHBOARD (modular)
# ─────────────────────────────────────────────────────────────────────────
elif selected_tab == "🛡 Admin":
    if not is_admin:
        st.error("🔒 Admin access required.")
    else:
        render_admin_dashboard(project="franchise")

# ─────────────────────────────────────────────────────────────────────────
# TAB: SETTINGS
# ─────────────────────────────────────────────────────────────────────────
elif selected_tab == "⚙ Settings":
    render_card('<h3 style="margin:0;">⚙ Settings</h3>')
    st.markdown(f'''
    <div class="glass-card">
        <p><b>Username:</b> {username}</p>
        <p><b>Role:</b> {user_role}</p>
        <p><b>LLM Engine:</b> Qwen-2.5-3B-Instruct (4-bit NF4)</p>
        <p><b>Theme:</b> Dark · Electric Blue / Cyan / Purple accents</p>
    </div>
    ''', unsafe_allow_html=True)

In [ ]:
!pip install -q streamlit pyngrok bcrypt pyjwt pandas numpy scikit-learn joblib transformers accelerate bitsandbytes plotly streamlit-option-menu faker kaggle

In [ ]:
!pip install streamlit
import subprocess, time, os
from pyngrok import ngrok
try:
    from config import NGROK_AUTHTOKEN as NGROK_AUTH_TOKEN
except ImportError:
    from config import NGROK_AUTH_TOKEN

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8501).public_url
    print("🚀 App Published at:", public_url)
else:
    print("Running locally on port 8501.")

process = subprocess.Popen(["streamlit", "run", "app.py",
                            "--server.port=8501", "--server.headless=true"])
print("✅ Streamlit started (PID:", process.pid, ")")

In [ ]:
try:
    process.terminate()
    ngrok.kill()
    print("🛑 Streamlit and ngrok terminated successfully.")
except Exception as e:
    print("Info:", e)